In [3]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 1: Build a seeded base table for the WebTable enrichment branch
# ------------------------------------------------------------
# Comments:
# - This table is the single input relation for the WebTable phase.
# - It starts from music_enrichment_base_for_webtables_v1.
# - We create one globally unique key per row so MB release IDs and
#   external residual IDs never collide.
# - We also create retrieval seeds:
#     * artist
#     * title
#     * year
# - For MusicBrainz rows, prefer MB fields.
# - For external residual rows, use the external fields.
# - We add normalized seed keys for later matching inside web tables.
print("Creating music_enrichment_base_for_webtables_seeded_v1...")

con.execute(r"""
    CREATE OR REPLACE TABLE music_enrichment_base_for_webtables_seeded_v1 AS
    SELECT
        *,

        -- ----------------------------------------------------
        -- Global unique key for the WebTable branch
        -- ----------------------------------------------------
        CASE
            WHEN base_record_status IN ('mb_release_enriched_from_cds', 'mb_release_not_enriched')
                THEN 'MB_' || CAST(release_id AS VARCHAR)
            WHEN base_record_status = 'external_residual'
                THEN 'EXT_' || CAST(external_record_id AS VARCHAR)
            ELSE 'UNK_' || CAST(base_record_id AS VARCHAR)
        END AS base_record_key,

        -- ----------------------------------------------------
        -- Seed fields for retrieval
        -- ----------------------------------------------------
        CASE
            WHEN base_record_status IN ('mb_release_enriched_from_cds', 'mb_release_not_enriched')
                THEN main_artist_name
            ELSE external_artist
        END AS seed_artist,

        CASE
            WHEN base_record_status IN ('mb_release_enriched_from_cds', 'mb_release_not_enriched')
                THEN release_title
            ELSE external_title
        END AS seed_title,

        CASE
            WHEN base_record_status IN ('mb_release_enriched_from_cds', 'mb_release_not_enriched')
                THEN mb_year
            ELSE external_year
        END AS seed_year,

        -- ----------------------------------------------------
        -- Clean text seeds
        -- ----------------------------------------------------
        trim(regexp_replace(
            lower(
                coalesce(
                    CASE
                        WHEN base_record_status IN ('mb_release_enriched_from_cds', 'mb_release_not_enriched')
                            THEN main_artist_name
                        ELSE external_artist
                    END,
                    ''
                )
            ),
            '\s+',
            ' ',
            'g'
        )) AS seed_artist_clean,

        trim(regexp_replace(
            lower(
                coalesce(
                    CASE
                        WHEN base_record_status IN ('mb_release_enriched_from_cds', 'mb_release_not_enriched')
                            THEN release_title
                        ELSE external_title
                    END,
                    ''
                )
            ),
            '\s+',
            ' ',
            'g'
        )) AS seed_title_clean,

        -- ----------------------------------------------------
        -- Compact alphanumeric matching keys
        -- ----------------------------------------------------
        lower(regexp_replace(
            coalesce(
                CASE
                    WHEN base_record_status IN ('mb_release_enriched_from_cds', 'mb_release_not_enriched')
                        THEN main_artist_name
                    ELSE external_artist
                END,
                ''
            ),
            '[^a-zA-Z0-9]+',
            '',
            'g'
        )) AS seed_artist_key,

        lower(regexp_replace(
            coalesce(
                CASE
                    WHEN base_record_status IN ('mb_release_enriched_from_cds', 'mb_release_not_enriched')
                        THEN release_title
                    ELSE external_title
                END,
                ''
            ),
            '[^a-zA-Z0-9]+',
            '',
            'g'
        )) AS seed_title_key,

        -- ----------------------------------------------------
        -- Retrieval query string
        -- ----------------------------------------------------
        trim(
            coalesce(
                CASE
                    WHEN base_record_status IN ('mb_release_enriched_from_cds', 'mb_release_not_enriched')
                        THEN main_artist_name
                    ELSE external_artist
                END,
                ''
            )
            || ' ' ||
            coalesce(
                CASE
                    WHEN base_record_status IN ('mb_release_enriched_from_cds', 'mb_release_not_enriched')
                        THEN release_title
                    ELSE external_title
                END,
                ''
            )
            || ' ' ||
            coalesce(
                CAST(
                    CASE
                        WHEN base_record_status IN ('mb_release_enriched_from_cds', 'mb_release_not_enriched')
                            THEN mb_year
                        ELSE external_year
                    END AS VARCHAR
                ),
                ''
            )
        ) AS seed_query

    FROM music_enrichment_base_for_webtables_v1
""")

print("✅ Table created: music_enrichment_base_for_webtables_seeded_v1")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM music_enrichment_base_for_webtables_seeded_v1
""").fetchdf())

print("\n--- Breakdown by base record status ---")
print(con.execute("""
    SELECT base_record_status, COUNT(*) AS n
    FROM music_enrichment_base_for_webtables_seeded_v1
    GROUP BY base_record_status
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        base_record_key,
        base_record_status,
        release_id,
        external_record_id,
        seed_artist,
        seed_title,
        seed_year,
        seed_query
    FROM music_enrichment_base_for_webtables_seeded_v1
    LIMIT 30
""").fetchdf())
con.close()
print("Connection closed ✅")

Connected ✅
Creating music_enrichment_base_for_webtables_seeded_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: music_enrichment_base_for_webtables_seeded_v1

--- Row count ---
         n
0  5314525

--- Breakdown by base record status ---
             base_record_status        n
0       mb_release_not_enriched  5305678
1             external_residual     4647
2  mb_release_enriched_from_cds     4200

--- Preview ---
   base_record_key       base_record_status  release_id  external_record_id  \
0       MB_2608361  mb_release_not_enriched     2608361                <NA>   
1       MB_2794510  mb_release_not_enriched     2794510                <NA>   
2       MB_2608364  mb_release_not_enriched     2608364                <NA>   
3       MB_2608363  mb_release_not_enriched     2608363                <NA>   
4       MB_2608362  mb_release_not_enriched     2608362                <NA>   
5       MB_1644640  mb_release_not_enriched     1644640                <NA>   
6        MB_463456  mb_release_not_enriched      463456                <NA>   
7       MB_2608365  mb_release_not_enriche

In [4]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 2: Define the target schema for WebTable enrichment candidates
# ------------------------------------------------------------
# Comments:
# - This table is intentionally created empty.
# - One row will later represent one candidate attribute value
#   extracted from one matched web table row.
# - We keep provenance, schema interpretation, and matching scores
#   in the same structure so the WebTable branch stays reproducible.
# - First target attributes will likely be:
#     * genre
#     * label
#     * year
#   but the schema is generic enough for more attributes later.
print("Creating music_webtable_enrichment_candidates_v1...")

con.execute("""
    CREATE OR REPLACE TABLE music_webtable_enrichment_candidates_v1 AS
    SELECT
        -- ----------------------------------------------------
        -- Base record identity
        -- ----------------------------------------------------
        NULL::VARCHAR AS base_record_key,
        NULL::VARCHAR AS base_record_status,
        NULL::BIGINT AS release_id,
        NULL::BIGINT AS external_record_id,

        -- ----------------------------------------------------
        -- Retrieval seed used for table / row search
        -- ----------------------------------------------------
        NULL::VARCHAR AS seed_artist,
        NULL::VARCHAR AS seed_title,
        NULL::INTEGER AS seed_year,

        -- ----------------------------------------------------
        -- Web table provenance
        -- ----------------------------------------------------
        NULL::VARCHAR AS webtable_id,
        NULL::VARCHAR AS source_url,
        NULL::VARCHAR AS page_title,
        NULL::VARCHAR AS table_caption,
        NULL::VARCHAR AS table_context_before,
        NULL::VARCHAR AS table_context_after,

        -- ----------------------------------------------------
        -- Schema interpretation
        -- ----------------------------------------------------
        NULL::VARCHAR AS artist_column_name,
        NULL::VARCHAR AS title_column_name,
        NULL::VARCHAR AS year_column_name,
        NULL::VARCHAR AS value_column_name,

        -- ----------------------------------------------------
        -- Matched row key / row identity inside the web table
        -- ----------------------------------------------------
        NULL::VARCHAR AS matched_row_key,

        -- ----------------------------------------------------
        -- Candidate enrichment fact
        -- ----------------------------------------------------
        NULL::VARCHAR AS attribute_name,
        NULL::VARCHAR AS candidate_value,

        -- ----------------------------------------------------
        -- Matching and evidence scores
        -- ----------------------------------------------------
        NULL::DOUBLE AS artist_similarity,
        NULL::DOUBLE AS title_similarity,
        NULL::DOUBLE AS year_compatibility,
        NULL::DOUBLE AS row_match_score,
        NULL::DOUBLE AS schema_match_score,
        NULL::DOUBLE AS evidence_score,

        -- ----------------------------------------------------
        -- Decision placeholders
        -- ----------------------------------------------------
        NULL::VARCHAR AS candidate_status,
        NULL::VARCHAR AS enrichment_source

    WHERE 1 = 0
""")

print("✅ Table created: music_webtable_enrichment_candidates_v1")

print("\n--- Schema preview ---")
print(con.execute("""
    PRAGMA table_info('music_webtable_enrichment_candidates_v1')
""").fetchdf())
con.close()
print("Connection closed ✅")

Connected ✅
Creating music_webtable_enrichment_candidates_v1...
✅ Table created: music_webtable_enrichment_candidates_v1

--- Schema preview ---
    cid                  name     type  notnull dflt_value     pk
0     0       base_record_key  VARCHAR    False       None  False
1     1    base_record_status  VARCHAR    False       None  False
2     2            release_id   BIGINT    False       None  False
3     3    external_record_id   BIGINT    False       None  False
4     4           seed_artist  VARCHAR    False       None  False
5     5            seed_title  VARCHAR    False       None  False
6     6             seed_year  INTEGER    False       None  False
7     7           webtable_id  VARCHAR    False       None  False
8     8            source_url  VARCHAR    False       None  False
9     9            page_title  VARCHAR    False       None  False
10   10         table_caption  VARCHAR    False       None  False
11   11  table_context_before  VARCHAR    False       None  Fal

In [5]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 3: Build a pilot subset for the WebTable branch
# ------------------------------------------------------------
# Comments:
# - The full base table is far too large for a first WebTable experiment.
# - So we create a balanced pilot sample from the three current groups:
#     * mb_release_enriched_from_cds
#     * mb_release_not_enriched
#     * external_residual
# - This pilot table will be the first input for retrieval, filtering,
#   schema matching, and row-level matching against WebTables.
# - Adjust the sample sizes if needed.
print("Creating music_enrichment_webtable_pilot_v1...")

N_ENRICHED = 100
N_NOT_ENRICHED = 100
N_EXTERNAL = 100

con.execute(f"""
    CREATE OR REPLACE TABLE music_enrichment_webtable_pilot_v1 AS

    SELECT *
    FROM (
        SELECT *
        FROM music_enrichment_base_for_webtables_seeded_v1
        WHERE base_record_status = 'mb_release_enriched_from_cds'
        ORDER BY base_record_key
        LIMIT {N_ENRICHED}
    )

    UNION ALL

    SELECT *
    FROM (
        SELECT *
        FROM music_enrichment_base_for_webtables_seeded_v1
        WHERE base_record_status = 'mb_release_not_enriched'
        ORDER BY base_record_key
        LIMIT {N_NOT_ENRICHED}
    )

    UNION ALL

    SELECT *
    FROM (
        SELECT *
        FROM music_enrichment_base_for_webtables_seeded_v1
        WHERE base_record_status = 'external_residual'
        ORDER BY base_record_key
        LIMIT {N_EXTERNAL}
    )
""")

print("✅ Table created: music_enrichment_webtable_pilot_v1")

print("\n--- Pilot size ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM music_enrichment_webtable_pilot_v1
""").fetchdf())

print("\n--- Pilot breakdown ---")
print(con.execute("""
    SELECT base_record_status, COUNT(*) AS n
    FROM music_enrichment_webtable_pilot_v1
    GROUP BY base_record_status
    ORDER BY n DESC
""").fetchdf())

print("\n--- Pilot preview ---")
print(con.execute("""
    SELECT
        base_record_key,
        base_record_status,
        seed_artist,
        seed_title,
        seed_year,
        seed_query
    FROM music_enrichment_webtable_pilot_v1
    LIMIT 30
""").fetchdf())

Connected ✅
Creating music_enrichment_webtable_pilot_v1...
✅ Table created: music_enrichment_webtable_pilot_v1

--- Pilot size ---
     n
0  300

--- Pilot breakdown ---
             base_record_status    n
0  mb_release_enriched_from_cds  100
1             external_residual  100
2       mb_release_not_enriched  100

--- Pilot preview ---
   base_record_key            base_record_status  \
0        MB_100020  mb_release_enriched_from_cds   
1       MB_1000444  mb_release_enriched_from_cds   
2       MB_1000479  mb_release_enriched_from_cds   
3       MB_1000797  mb_release_enriched_from_cds   
4       MB_1000814  mb_release_enriched_from_cds   
5       MB_1000937  mb_release_enriched_from_cds   
6       MB_1001030  mb_release_enriched_from_cds   
7       MB_1001088  mb_release_enriched_from_cds   
8        MB_100143  mb_release_enriched_from_cds   
9        MB_100207  mb_release_enriched_from_cds   
10      MB_1002185  mb_release_enriched_from_cds   
11      MB_1002603  mb_release_enri

In [24]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 4: Create raw staging table for WebTables ingestion
# ------------------------------------------------------------
# Comments:
# - This is a raw landing table for WebTable records.
# - It stays generic because the exact WDC file format can vary depending
#   on which release / extraction file you use.
# - For now, the table is intentionally empty.
# - Later, after downloading a manageable WebTables subset, you can insert
#   raw rows here before music-table filtering.
print("Creating wdc_webtables_raw_v1...")

con.execute("""
    CREATE OR REPLACE TABLE wdc_webtables_raw_v1 AS
    SELECT
        -- ----------------------------------------------------
        -- Web table identity / provenance
        -- ----------------------------------------------------
        NULL::VARCHAR AS webtable_id,
        NULL::VARCHAR AS source_url,
        NULL::VARCHAR AS page_title,
        NULL::VARCHAR AS table_caption,

        -- ----------------------------------------------------
        -- Context around the table
        -- ----------------------------------------------------
        NULL::VARCHAR AS table_context_before,
        NULL::VARCHAR AS table_context_after,

        -- ----------------------------------------------------
        -- Structural metadata
        -- ----------------------------------------------------
        NULL::VARCHAR AS orientation,
        NULL::BOOLEAN AS has_header,
        NULL::INTEGER AS key_column_index,

        -- ----------------------------------------------------
        -- Raw extracted content
        -- relation_json:
        --   original raw table content, e.g. as JSON string
        -- header_json:
        --   optional explicit header representation
        -- ----------------------------------------------------
        NULL::VARCHAR AS relation_json,
        NULL::VARCHAR AS header_json,

        -- ----------------------------------------------------
        -- Optional source-level notes
        -- ----------------------------------------------------
        NULL::VARCHAR AS source_domain,
        NULL::VARCHAR AS ingest_note

    WHERE 1 = 0
""")

print("✅ Table created: wdc_webtables_raw_v1")

print("\n--- Schema preview ---")
print(con.execute("""
    PRAGMA table_info('wdc_webtables_raw_v1')
""").fetchdf())

Connected ✅
Creating wdc_webtables_raw_v1...
✅ Table created: wdc_webtables_raw_v1

--- Schema preview ---
    cid                  name     type  notnull dflt_value     pk
0     0           webtable_id  VARCHAR    False       None  False
1     1            source_url  VARCHAR    False       None  False
2     2            page_title  VARCHAR    False       None  False
3     3         table_caption  VARCHAR    False       None  False
4     4  table_context_before  VARCHAR    False       None  False
5     5   table_context_after  VARCHAR    False       None  False
6     6           orientation  VARCHAR    False       None  False
7     7            has_header  BOOLEAN    False       None  False
8     8      key_column_index  INTEGER    False       None  False
9     9         relation_json  VARCHAR    False       None  False
10   10           header_json  VARCHAR    False       None  False
11   11         source_domain  VARCHAR    False       None  False
12   12           ingest_note  VARC

In [25]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

sample_path = "../data_raw/webtables_sample/sample/sample"

# ------------------------------------------------------------
# Step 5: Load raw WDC sample into DuckDB
# ------------------------------------------------------------
# Comments:
# - The sample file is line-delimited JSON:
#   one web table = one JSON object per line
# - We load the raw JSON fields first, without forcing music logic yet
# - relation_json stores the raw relation array as text for later parsing
print("Creating and loading wdc_webtables_raw_v1...")

con.execute(f"""
    CREATE OR REPLACE TABLE wdc_webtables_raw_v1 AS
    SELECT
        -- ----------------------------------------------------
        -- Build a local webtable id
        -- ----------------------------------------------------
        'WT_' || CAST(row_number() OVER () AS VARCHAR) AS webtable_id,

        url AS source_url,
        pageTitle AS page_title,
        title AS table_caption,
        textBeforeTable AS table_context_before,
        textAfterTable AS table_context_after,

        tableOrientation AS orientation,
        hasHeader AS has_header,
        keyColumnIndex AS key_column_index,

        -- store raw content as JSON text for later parsing
        CAST(relation AS VARCHAR) AS relation_json,

        -- store optional header metadata as JSON-like text
        CAST(headerPosition AS VARCHAR) AS header_json,

        -- extra useful raw fields
        tableType AS table_type,
        hasKeyColumn AS has_key_column,
        headerRowIndex AS header_row_index,
        tableNum AS table_num,
        s3Link AS s3_link,
        recordOffset AS record_offset,
        recordEndOffset AS record_end_offset,

        regexp_extract(url, 'https?://([^/]+)', 1) AS source_domain,
        'wdc_sample_2015' AS ingest_note

    FROM read_json_auto('{sample_path}', format='newline_delimited')
""")

print("✅ Table created: wdc_webtables_raw_v1")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_webtables_raw_v1
""").fetchdf())

print("\n--- Table type breakdown ---")
print(con.execute("""
    SELECT table_type, COUNT(*) AS n
    FROM wdc_webtables_raw_v1
    GROUP BY table_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Orientation breakdown ---")
print(con.execute("""
    SELECT orientation, COUNT(*) AS n
    FROM wdc_webtables_raw_v1
    GROUP BY orientation
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        table_type,
        orientation,
        has_header,
        has_key_column,
        page_title,
        table_caption
    FROM wdc_webtables_raw_v1
    LIMIT 20
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating and loading wdc_webtables_raw_v1...
✅ Table created: wdc_webtables_raw_v1

--- Row count ---
      n
0  9275

--- Table type breakdown ---
  table_type     n
0     ENTITY  4732
1   RELATION  2740
2     LAYOUT  1362
3      OTHER   357
4     MATRIX    84

--- Orientation breakdown ---
  orientation     n
0  HORIZONTAL  6265
1    VERTICAL  2927
2       MIXED    83

--- Preview ---
   webtable_id                source_domain table_type orientation  \
0         WT_1                     1980s.fm     LAYOUT  HORIZONTAL   
1         WT_2                     1980s.fm     LAYOUT  HORIZONTAL   
2         WT_3                  3docean.net     ENTITY    VERTICAL   
3         WT_4         67-72chevytrucks.com     LAYOUT    VERTICAL   
4         WT_5                  965kvki.com   RELATION  HORIZONTAL   
5         WT_6      Gayle.Haarsma@dordt.edu   RELATION  HORIZONTAL   
6         WT_7             abacus.bates.edu   RELATION  HORIZONTAL   
7         WT_8                accleagu

In [13]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 6: Filter raw WebTables to likely music-related candidate tables
# ------------------------------------------------------------
# Comments:
# - We remove obvious layout noise first
# - We prefer RELATION and ENTITY tables
# - We keep tables whose page/caption/context/relation text
#   contains music-related cues
# - This is still a broad candidate set, not final matching
print("Creating wdc_music_tables_candidate_v1...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_tables_candidate_v1 AS
    WITH base AS (
        SELECT
            *,
            lower(coalesce(page_title, '')) AS page_title_lc,
            lower(coalesce(table_caption, '')) AS table_caption_lc,
            lower(coalesce(table_context_before, '')) AS ctx_before_lc,
            lower(coalesce(table_context_after, '')) AS ctx_after_lc,
            lower(coalesce(relation_json, '')) AS relation_lc
        FROM wdc_webtables_raw_v1
    )
    SELECT
        *
    FROM base
    WHERE
        -- ----------------------------------------------------
        -- Prefer structured tables over layout tables
        -- ----------------------------------------------------
        table_type IN ('RELATION', 'ENTITY')

        -- ----------------------------------------------------
        -- Keep tables with music-like signals
        -- ----------------------------------------------------
        AND (
            regexp_matches(page_title_lc,
                'artist|album|song|songs|track|tracks|discography|music|musician|band|record|records|label|release|releases|vinyl|cd|disc')
            OR regexp_matches(table_caption_lc,
                'artist|album|song|songs|track|tracks|discography|music|band|label|release|vinyl|cd')
            OR regexp_matches(ctx_before_lc,
                'artist|album|song|songs|track|tracks|discography|music|band|label|release')
            OR regexp_matches(ctx_after_lc,
                'artist|album|song|songs|track|tracks|discography|music|band|label|release')
            OR regexp_matches(relation_lc,
                'artist|album|song|songs|track|tracks|discography|music|band|label|release|genre|year')
        )
""")

print("✅ Table created: wdc_music_tables_candidate_v1")

print("\n--- Candidate count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_tables_candidate_v1
""").fetchdf())

print("\n--- Table type breakdown ---")
print(con.execute("""
    SELECT table_type, COUNT(*) AS n
    FROM wdc_music_tables_candidate_v1
    GROUP BY table_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Source domain preview ---")
print(con.execute("""
    SELECT source_domain, COUNT(*) AS n
    FROM wdc_music_tables_candidate_v1
    GROUP BY source_domain
    ORDER BY n DESC
    LIMIT 20
""").fetchdf())

print("\n--- Preview of music-like candidate tables ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        table_type,
        orientation,
        page_title,
        table_caption
    FROM wdc_music_tables_candidate_v1
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_tables_candidate_v1...
✅ Table created: wdc_music_tables_candidate_v1

--- Candidate count ---
      n
0  1952

--- Table type breakdown ---
  table_type     n
0     ENTITY  1178
1   RELATION   774

--- Source domain preview ---
            source_domain    n
0   www.peoplefinders.com  157
1    www.wincustomize.com   52
2           gtaforums.com   40
3          www.theday.com   40
4    www.healthgrades.com   40
5        wincustomize.com   39
6      www.summitpost.org   36
7            seatgeek.com   30
8      clinicaltrials.gov   20
9          www.cappex.com   19
10     www.metalstorm.net   18
11         eprints.ucm.es   18
12     www.1080thefan.com   18
13      stackoverflow.com   16
14               obra.org   15
15       itunes.apple.com   14
16  www.bigstockphoto.com   14
17       www.dpreview.com   14
18           www.gilt.com   13
19       www.worldcat.org   13

--- Preview of music-like candidate tables ---
   webtable_id                     source

In [14]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 7: Build a stricter music-table candidate set
# ------------------------------------------------------------
# Comments:
# - The first candidate table is still too noisy.
# - Here we tighten the filter:
#     * keep only RELATION / ENTITY tables
#     * require stronger music cues
#     * exclude obviously irrelevant domains
#     * avoid generic false positives such as "sound" alone
print("Creating wdc_music_tables_candidate_strict_v1...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_tables_candidate_strict_v1 AS
    WITH base AS (
        SELECT
            *,
            lower(coalesce(source_domain, '')) AS domain_lc,
            lower(coalesce(page_title, '')) AS page_title_lc,
            lower(coalesce(table_caption, '')) AS table_caption_lc,
            lower(coalesce(table_context_before, '')) AS ctx_before_lc,
            lower(coalesce(table_context_after, '')) AS ctx_after_lc,
            lower(coalesce(relation_json, '')) AS relation_lc
        FROM wdc_webtables_raw_v1
    )
    SELECT *
    FROM base
    WHERE
        table_type IN ('RELATION', 'ENTITY')

        -- strong positive music signals
        AND (
            regexp_matches(page_title_lc,
                'album|albums|discography|tracklist|track list|record label|music by|song|songs|single|singles|ep|lp|vinyl|cd|disc|artist|artists|band|bands|musician|musicians|soundtrack|release|releases|genre')
            OR regexp_matches(table_caption_lc,
                'album|albums|discography|tracklist|track list|record label|music by|song|songs|single|singles|ep|lp|vinyl|cd|disc|artist|artists|band|bands|soundtrack|release|genre')
            OR regexp_matches(relation_lc,
                'album|albums|track|tracks|tracklist|genre|label|artist|release|releases|discography|soundtrack')
        )

        -- remove obvious non-music domains / false-positive contexts
        AND NOT regexp_matches(domain_lc,
            'peoplefinders|healthgrades|stackoverflow|stackexchange|bccondos|clinicaltrials|cappex|dpreview|gilt|water\\.ca\\.gov|fantasysports|worldcat')

        -- remove obvious non-music page themes
        AND NOT regexp_matches(page_title_lc,
            'box office|baseball|wiring diagram|materials and shaders|cross country|clinical trial|condo|wedding band|forum post|statistics')
""")

print("✅ Table created: wdc_music_tables_candidate_strict_v1")

print("\n--- Strict candidate count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_tables_candidate_strict_v1
""").fetchdf())

print("\n--- Table type breakdown ---")
print(con.execute("""
    SELECT table_type, COUNT(*) AS n
    FROM wdc_music_tables_candidate_strict_v1
    GROUP BY table_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Source domain preview ---")
print(con.execute("""
    SELECT source_domain, COUNT(*) AS n
    FROM wdc_music_tables_candidate_strict_v1
    GROUP BY source_domain
    ORDER BY n DESC
    LIMIT 20
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        table_type,
        orientation,
        page_title
    FROM wdc_music_tables_candidate_strict_v1
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_tables_candidate_strict_v1...
✅ Table created: wdc_music_tables_candidate_strict_v1

--- Strict candidate count ---
     n
0  935

--- Table type breakdown ---
  table_type    n
0     ENTITY  547
1   RELATION  388

--- Source domain preview ---
                source_domain   n
0                www.reef.org  76
1        www.wincustomize.com  58
2            wincustomize.com  39
3               gtaforums.com  24
4         de.magicseaweed.com  18
5            magicseaweed.com  17
6            itunes.apple.com  14
7         speedwaysonline.com  10
8              www.theday.com  10
9   www.mcdanielathletics.com  10
10        www.cpa.state.tx.us  10
11      www.bigstockphoto.com   9
12         www.metalstorm.net   9
13                www.unm.edu   8
14           httpd.apache.org   7
15               data.bls.gov   7
16          www.bluesnews.com   7
17         www.summitpost.org   6
18             eprints.ucm.es   6
19            flightaware.com   6

--- Previ

In [15]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 8: Find strict candidate tables that match pilot record seeds
# ------------------------------------------------------------
# Comments:
# - This is the first record-aware retrieval step.
# - We compare pilot seed artist/title strings against table text.
# - We do NOT parse row structure yet.
# - Goal:
#   keep only tables that are plausible evidence for at least one pilot record.
print("Creating wdc_music_table_hits_for_pilot_v1...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_table_hits_for_pilot_v1 AS
    WITH tables AS (
        SELECT
            webtable_id,
            source_domain,
            table_type,
            orientation,
            page_title,
            table_caption,
            table_context_before,
            table_context_after,
            relation_json,
            lower(
                coalesce(page_title, '') || ' ' ||
                coalesce(table_caption, '') || ' ' ||
                coalesce(table_context_before, '') || ' ' ||
                coalesce(table_context_after, '') || ' ' ||
                coalesce(relation_json, '')
            ) AS table_text_lc
        FROM wdc_music_tables_candidate_strict_v1
    ),
    pilot AS (
        SELECT
            base_record_key,
            base_record_status,
            seed_artist,
            seed_title,
            seed_year,
            lower(coalesce(seed_artist, '')) AS seed_artist_lc,
            lower(coalesce(seed_title, '')) AS seed_title_lc
        FROM music_enrichment_webtable_pilot_v1
        WHERE
            seed_artist IS NOT NULL AND trim(seed_artist) <> ''
            AND seed_title IS NOT NULL AND trim(seed_title) <> ''
    )
    SELECT
        p.base_record_key,
        p.base_record_status,
        p.seed_artist,
        p.seed_title,
        p.seed_year,
        t.webtable_id,
        t.source_domain,
        t.table_type,
        t.orientation,
        t.page_title,
        t.table_caption,

        CASE
            WHEN strpos(t.table_text_lc, p.seed_artist_lc) > 0 THEN 1 ELSE 0
        END AS artist_hit,

        CASE
            WHEN strpos(t.table_text_lc, p.seed_title_lc) > 0 THEN 1 ELSE 0
        END AS title_hit,

        CASE
            WHEN p.seed_year IS NOT NULL
                 AND strpos(t.table_text_lc, CAST(p.seed_year AS VARCHAR)) > 0
            THEN 1 ELSE 0
        END AS year_hit

    FROM pilot p
    JOIN tables t
      ON (
           strpos(t.table_text_lc, p.seed_artist_lc) > 0
           OR strpos(t.table_text_lc, p.seed_title_lc) > 0
         )
    WHERE
        -- keep only stronger hits
        (
            (strpos(t.table_text_lc, p.seed_artist_lc) > 0 AND strpos(t.table_text_lc, p.seed_title_lc) > 0)
            OR
            (strpos(t.table_text_lc, p.seed_title_lc) > 0 AND p.seed_year IS NOT NULL AND strpos(t.table_text_lc, CAST(p.seed_year AS VARCHAR)) > 0)
        )
""")

print("✅ Table created: wdc_music_table_hits_for_pilot_v1")

print("\n--- Hit count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_hits_for_pilot_v1
""").fetchdf())

print("\n--- Distinct pilot records with at least one table hit ---")
print(con.execute("""
    SELECT COUNT(DISTINCT base_record_key) AS n
    FROM wdc_music_table_hits_for_pilot_v1
""").fetchdf())

print("\n--- Distinct candidate tables hit by pilot records ---")
print(con.execute("""
    SELECT COUNT(DISTINCT webtable_id) AS n
    FROM wdc_music_table_hits_for_pilot_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        base_record_key,
        seed_artist,
        seed_title,
        seed_year,
        webtable_id,
        source_domain,
        page_title,
        artist_hit,
        title_hit,
        year_hit
    FROM wdc_music_table_hits_for_pilot_v1
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_hits_for_pilot_v1...
✅ Table created: wdc_music_table_hits_for_pilot_v1

--- Hit count ---
    n
0  19

--- Distinct pilot records with at least one table hit ---
   n
0  6

--- Distinct candidate tables hit by pilot records ---
    n
0  17

--- Preview ---
   base_record_key        seed_artist seed_title  seed_year webtable_id  \
0       MB_1005722       Otis Redding       Live       2019     WT_7579   
1       MB_1005722       Otis Redding       Live       2019     WT_8889   
2        MB_100846         Bill Evans      Alone       2010     WT_2287   
3       MB_1010729  Celtic Connection     Higher       2025       WT_52   
4       MB_1000008        Nação Zumbi     Futura       2013     WT_8889   
5       EXT_100050            phallus     reason       2003      WT_444   
6       EXT_100050            phallus     reason       2003     WT_2287   
7       EXT_100070        perseguidos        iii       2004      WT_278   
8       EXT_100070        pers

In [17]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 9: Build a stricter pilot-to-webtable hit table
# ------------------------------------------------------------
# Comments:
# - The previous hit table was too noisy because many matches came from
#   weak title-only hits such as "Live", "Alone", or "III".
# - Here we enforce stronger retrieval logic:
#     * prefer artist + title hits
#     * allow title + year only for stronger titles
#     * explicitly downweight generic or short titles
print("Creating wdc_music_table_hits_for_pilot_strict_v2...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_table_hits_for_pilot_strict_v2 AS
    WITH tables AS (
        SELECT
            webtable_id,
            source_domain,
            table_type,
            orientation,
            page_title,
            table_caption,
            table_context_before,
            table_context_after,
            relation_json,
            lower(
                coalesce(page_title, '') || ' ' ||
                coalesce(table_caption, '') || ' ' ||
                coalesce(table_context_before, '') || ' ' ||
                coalesce(table_context_after, '') || ' ' ||
                coalesce(relation_json, '')
            ) AS table_text_lc
        FROM wdc_music_tables_candidate_strict_v1
    ),
    pilot AS (
        SELECT
            base_record_key,
            base_record_status,
            seed_artist,
            seed_title,
            seed_year,
            lower(trim(coalesce(seed_artist, ''))) AS seed_artist_lc,
            lower(trim(coalesce(seed_title, ''))) AS seed_title_lc,
            length(trim(coalesce(seed_title, ''))) AS seed_title_len,

            CASE
                WHEN lower(trim(coalesce(seed_title, ''))) IN (
                    'live', 'alone', 'higher', 'song', 'songs',
                    'greatest hits', 'best of', 'the best of',
                    'hits', 'music', 'reason'
                ) THEN 1
                WHEN regexp_matches(lower(trim(coalesce(seed_title, ''))), '^(i|ii|iii|iv|v|vi|vii|viii|ix|x)$') THEN 1
                WHEN length(trim(coalesce(seed_title, ''))) <= 3 THEN 1
                ELSE 0
            END AS is_weak_title
        FROM music_enrichment_webtable_pilot_v1
        WHERE
            seed_artist IS NOT NULL AND trim(seed_artist) <> ''
            AND seed_title IS NOT NULL AND trim(seed_title) <> ''
    ),
    matched AS (
        SELECT
            p.base_record_key,
            p.base_record_status,
            p.seed_artist,
            p.seed_title,
            p.seed_year,
            p.seed_title_len,
            p.is_weak_title,

            t.webtable_id,
            t.source_domain,
            t.table_type,
            t.orientation,
            t.page_title,
            t.table_caption,

            CASE
                WHEN strpos(t.table_text_lc, p.seed_artist_lc) > 0 THEN 1 ELSE 0
            END AS artist_hit,

            CASE
                WHEN strpos(t.table_text_lc, p.seed_title_lc) > 0 THEN 1 ELSE 0
            END AS title_hit,

            CASE
                WHEN p.seed_year IS NOT NULL
                     AND strpos(t.table_text_lc, CAST(p.seed_year AS VARCHAR)) > 0
                THEN 1 ELSE 0
            END AS year_hit
        FROM pilot p
        JOIN tables t
          ON (
               strpos(t.table_text_lc, p.seed_artist_lc) > 0
               OR strpos(t.table_text_lc, p.seed_title_lc) > 0
             )
    )
    SELECT
        *,
        (0.5 * artist_hit + 0.4 * title_hit + 0.1 * year_hit) AS retrieval_score
    FROM matched
    WHERE
        (
            -- strongest case
            artist_hit = 1 AND title_hit = 1
        )
        OR
        (
            -- allow artist + year for weak titles
            artist_hit = 1 AND year_hit = 1
        )
        OR
        (
            -- allow title + year only if title is not weak
            title_hit = 1 AND year_hit = 1 AND is_weak_title = 0 AND seed_title_len >= 5
        )
""")

print("✅ Table created: wdc_music_table_hits_for_pilot_strict_v2")

print("\n--- Strict hit count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_hits_for_pilot_strict_v2
""").fetchdf())

print("\n--- Distinct pilot records with hits ---")
print(con.execute("""
    SELECT COUNT(DISTINCT base_record_key) AS n
    FROM wdc_music_table_hits_for_pilot_strict_v2
""").fetchdf())

print("\n--- Distinct candidate tables hit ---")
print(con.execute("""
    SELECT COUNT(DISTINCT webtable_id) AS n
    FROM wdc_music_table_hits_for_pilot_strict_v2
""").fetchdf())

print("\n--- Hit pattern breakdown ---")
print(con.execute("""
    SELECT
        artist_hit,
        title_hit,
        year_hit,
        COUNT(*) AS n
    FROM wdc_music_table_hits_for_pilot_strict_v2
    GROUP BY artist_hit, title_hit, year_hit
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        base_record_key,
        seed_artist,
        seed_title,
        seed_year,
        webtable_id,
        source_domain,
        page_title,
        artist_hit,
        title_hit,
        year_hit,
        is_weak_title,
        retrieval_score
    FROM wdc_music_table_hits_for_pilot_strict_v2
    ORDER BY retrieval_score DESC, base_record_key
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_hits_for_pilot_strict_v2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: wdc_music_table_hits_for_pilot_strict_v2

--- Strict hit count ---
    n
0  25

--- Distinct pilot records with hits ---
    n
0  12

--- Distinct candidate tables hit ---
    n
0  15

--- Hit pattern breakdown ---
   artist_hit  title_hit  year_hit   n
0           1          0         1  24
1           0          1         1   1

--- Preview ---
   base_record_key      seed_artist                         seed_title  \
0       EXT_100004          emanuel                         felicidade   
1       EXT_100017          various           micmac house dance party   
2       EXT_100017          various           micmac house dance party   
3       EXT_100166             vari                  el campeon latino   
4       EXT_100166             vari                  el campeon latino   
5       EXT_100166             vari                  el campeon latino   
6       EXT_100166             vari                  el campeon latino   
7       EXT_100166             vari       

In [18]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 10: Build a stronger pilot subset for row-level WebTable matching
# ------------------------------------------------------------
# Comments:
# - The earlier pilot still contained many weak seeds such as:
#     * Various Artists
#     * very short artist names
#     * generic titles like "Live"
#     * Roman numerals / weak one-word titles
# - For the first real row-level experiment, we keep only stronger,
#   more discriminative records.
# - This is a thesis-safe pilot reduction, not a permanent exclusion rule.
print("Creating music_enrichment_webtable_pilot_strong_v1...")

con.execute(r"""
    CREATE OR REPLACE TABLE music_enrichment_webtable_pilot_strong_v1 AS
    WITH base AS (
        SELECT
            *,
            lower(trim(coalesce(seed_artist, ''))) AS seed_artist_lc,
            lower(trim(coalesce(seed_title, ''))) AS seed_title_lc,
            length(trim(coalesce(seed_artist, ''))) AS seed_artist_len,
            length(trim(coalesce(seed_title, ''))) AS seed_title_len
        FROM music_enrichment_webtable_pilot_v1
    )
    SELECT *
    FROM base
    WHERE
        -- ----------------------------------------------------
        -- Keep readable, minimally informative seeds
        -- ----------------------------------------------------
        seed_artist IS NOT NULL
        AND trim(seed_artist) <> ''
        AND seed_title IS NOT NULL
        AND trim(seed_title) <> ''
        AND seed_artist_len >= 4
        AND seed_title_len >= 5

        -- ----------------------------------------------------
        -- Exclude overly generic artist placeholders
        -- ----------------------------------------------------
        AND seed_artist_lc NOT IN (
            'various', 'various artists', 'vari', 'va', 'v a'
        )

        -- ----------------------------------------------------
        -- Exclude generic / weak titles for the first row-level pilot
        -- ----------------------------------------------------
        AND seed_title_lc NOT IN (
            'live', 'alone', 'higher', 'song', 'songs',
            'greatest hits', 'best of', 'the best of',
            'hits', 'music', 'reason'
        )

        -- Exclude pure Roman numerals / ultra-weak tokens
        AND NOT regexp_matches(seed_title_lc, '^(i|ii|iii|iv|v|vi|vii|viii|ix|x)$')

        -- Exclude purely numeric titles for now
        AND NOT regexp_matches(seed_title_lc, '^[0-9]+$')
""")

print("✅ Table created: music_enrichment_webtable_pilot_strong_v1")

print("\n--- Strong pilot size ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM music_enrichment_webtable_pilot_strong_v1
""").fetchdf())

print("\n--- Strong pilot breakdown ---")
print(con.execute("""
    SELECT base_record_status, COUNT(*) AS n
    FROM music_enrichment_webtable_pilot_strong_v1
    GROUP BY base_record_status
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        base_record_key,
        base_record_status,
        seed_artist,
        seed_title,
        seed_year
    FROM music_enrichment_webtable_pilot_strong_v1
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating music_enrichment_webtable_pilot_strong_v1...
✅ Table created: music_enrichment_webtable_pilot_strong_v1

--- Strong pilot size ---
     n
0  249

--- Strong pilot breakdown ---
             base_record_status   n
0  mb_release_enriched_from_cds  90
1             external_residual  85
2       mb_release_not_enriched  74

--- Preview ---
   base_record_key            base_record_status  \
0        MB_100020  mb_release_enriched_from_cds   
1       MB_1000444  mb_release_enriched_from_cds   
2       MB_1000479  mb_release_enriched_from_cds   
3       MB_1000797  mb_release_enriched_from_cds   
4       MB_1000814  mb_release_enriched_from_cds   
5       MB_1000937  mb_release_enriched_from_cds   
6       MB_1001030  mb_release_enriched_from_cds   
7        MB_100143  mb_release_enriched_from_cds   
8        MB_100207  mb_release_enriched_from_cds   
9       MB_1002603  mb_release_enriched_from_cds   
10      MB_1002731  mb_release_enriched_from_cds   
11      MB_100283

In [19]:
import duckdb
import pandas as pd
import json

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 11: Parse relation_json into row-level WebTable records
# ------------------------------------------------------------
# Comments:
# - WDC stores the raw table body in the "relation" field.
# - We parse relation_json from the strict candidate set and create
#   one row per logical table row.
# - This table will later support:
#     * schema matching
#     * row-level entity matching
#     * candidate value extraction
# - Orientation handling:
#     * HORIZONTAL: relation is already row-oriented enough for a first pass
#       after simple normalization
#     * VERTICAL: transpose first so that table rows become comparable records
# - We also preserve a compact row_text field for first-stage matching.
print("Loading strict candidate tables for relation parsing...")

df = con.execute("""
    SELECT
        webtable_id,
        source_domain,
        table_type,
        orientation,
        has_header,
        key_column_index,
        page_title,
        table_caption,
        relation_json
    FROM wdc_music_tables_candidate_strict_v1
""").fetchdf()

print(f"Loaded candidate tables: {len(df):,}")

parsed_rows = []

def safe_str(x):
    return "" if x is None else str(x).strip()

def transpose_if_needed(matrix):
    if not matrix:
        return []
    max_len = max(len(r) for r in matrix)
    padded = [r + [""] * (max_len - len(r)) for r in matrix]
    return [list(col) for col in zip(*padded)]

for _, rec in df.iterrows():
    webtable_id = rec["webtable_id"]
    source_domain = rec["source_domain"]
    table_type = rec["table_type"]
    orientation = safe_str(rec["orientation"]).upper()
    has_header = rec["has_header"]
    key_column_index = rec["key_column_index"]
    page_title = rec["page_title"]
    table_caption = rec["table_caption"]
    relation_json = rec["relation_json"]

    try:
        relation = json.loads(relation_json)
    except Exception:
        continue

    if not isinstance(relation, list) or len(relation) == 0:
        continue

    # Normalize each row/cell to strings
    matrix = []
    for row in relation:
        if isinstance(row, list):
            matrix.append([safe_str(cell) for cell in row])

    if not matrix:
        continue

    # For vertical tables, transpose so that downstream row-level matching
    # sees row-like records instead of attribute stacks
    if orientation == "VERTICAL":
        matrix = transpose_if_needed(matrix)

    # Header handling
    header = None
    data_rows = matrix

    if bool(has_header) and len(matrix) >= 1:
        header = matrix[0]
        data_rows = matrix[1:]

    # Parse data rows
    for row_idx, row in enumerate(data_rows):
        row_cells = [safe_str(x) for x in row]
        nonempty_cells = [x for x in row_cells if x != ""]

        if len(nonempty_cells) == 0:
            continue

        row_text = " | ".join(nonempty_cells)
        row_text_lc = row_text.lower()

        # Optional row key from key column if available
        row_key = None
        try:
            if key_column_index is not None and int(key_column_index) >= 0 and int(key_column_index) < len(row_cells):
                row_key = safe_str(row_cells[int(key_column_index)])
        except Exception:
            row_key = None

        parsed_rows.append({
            "webtable_id": webtable_id,
            "source_domain": source_domain,
            "table_type": table_type,
            "orientation": orientation,
            "has_header": bool(has_header) if pd.notna(has_header) else None,
            "key_column_index": int(key_column_index) if pd.notna(key_column_index) else None,
            "page_title": page_title,
            "table_caption": table_caption,
            "header_json": json.dumps(header, ensure_ascii=False) if header is not None else None,
            "row_index": row_idx,
            "row_key": row_key,
            "row_json": json.dumps(row_cells, ensure_ascii=False),
            "row_text": row_text,
            "row_text_lc": row_text_lc,
            "cell_count": len(row_cells),
            "nonempty_cell_count": len(nonempty_cells)
        })

rows_df = pd.DataFrame(parsed_rows)

print(f"Parsed table rows: {len(rows_df):,}")

if rows_df.empty:
    print("⚠️ No rows were parsed. Check relation_json format.")
else:
    con.register("wdc_rows_df_view", rows_df)

    con.execute("""
        CREATE OR REPLACE TABLE wdc_music_table_rows_v1 AS
        SELECT *
        FROM wdc_rows_df_view
    """)

    print("✅ Table created: wdc_music_table_rows_v1")

    print("\n--- Row count ---")
    print(con.execute("""
        SELECT COUNT(*) AS n
        FROM wdc_music_table_rows_v1
    """).fetchdf())

    print("\n--- Distinct source tables ---")
    print(con.execute("""
        SELECT COUNT(DISTINCT webtable_id) AS n
        FROM wdc_music_table_rows_v1
    """).fetchdf())

    print("\n--- Orientation breakdown ---")
    print(con.execute("""
        SELECT orientation, COUNT(*) AS n
        FROM wdc_music_table_rows_v1
        GROUP BY orientation
        ORDER BY n DESC
    """).fetchdf())

    print("\n--- Preview ---")
    print(con.execute("""
        SELECT
            webtable_id,
            source_domain,
            table_type,
            orientation,
            row_index,
            row_key,
            row_text,
            cell_count,
            nonempty_cell_count
        FROM wdc_music_table_rows_v1
        LIMIT 30
    """).fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Loading strict candidate tables for relation parsing...
Loaded candidate tables: 935
Parsed table rows: 0
⚠️ No rows were parsed. Check relation_json format.

Connection closed ✅


In [20]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT
        webtable_id,
        relation_json
    FROM wdc_music_tables_candidate_strict_v1
    LIMIT 5
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
  webtable_id                                      relation_json
0       WT_30  [[hd, duration, published, description, name, ...
1       WT_39  [[Created, Audio Files Included, Bit Rate, Sam...
2       WT_40  [[Created, Looped Audio, Audio Files Included,...
3       WT_41  [[Created, Looped Audio, Audio Files Included,...
4       WT_42  [[Created, Looped Audio, Audio Files Included,...

Connection closed ✅


In [21]:
import duckdb
import pandas as pd
import json
import ast

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 11 (fixed): Parse relation_json into row-level WebTable records
# ------------------------------------------------------------
# Comments:
# - The previous attempt returned 0 rows because relation_json as stored
#   in DuckDB was not always parseable with json.loads() alone.
# - Here we use a more robust parser:
#     1) if already a Python list -> use directly
#     2) try json.loads()
#     3) fallback to ast.literal_eval()
# - We then convert each table into row-level records.
print("Loading strict candidate tables for robust relation parsing...")

df = con.execute("""
    SELECT
        webtable_id,
        source_domain,
        table_type,
        orientation,
        has_header,
        key_column_index,
        page_title,
        table_caption,
        relation_json
    FROM wdc_music_tables_candidate_strict_v1
""").fetchdf()

print(f"Loaded candidate tables: {len(df):,}")

parsed_rows = []

def safe_str(x):
    return "" if x is None else str(x).strip()

def transpose_if_needed(matrix):
    if not matrix:
        return []
    max_len = max(len(r) for r in matrix)
    padded = [r + [""] * (max_len - len(r)) for r in matrix]
    return [list(col) for col in zip(*padded)]

def parse_relation(value):
    # already a list-like object
    if isinstance(value, list):
        return value

    # pandas may hold arrays/objects that stringify weirdly
    if pd.isna(value):
        return None

    # if it is bytes
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="ignore")

    # first try json
    try:
        return json.loads(value)
    except Exception:
        pass

    # fallback: Python literal parser
    try:
        return ast.literal_eval(value)
    except Exception:
        pass

    return None

failed_examples = []

for _, rec in df.iterrows():
    webtable_id = rec["webtable_id"]
    source_domain = rec["source_domain"]
    table_type = rec["table_type"]
    orientation = safe_str(rec["orientation"]).upper()
    has_header = rec["has_header"]
    key_column_index = rec["key_column_index"]
    page_title = rec["page_title"]
    table_caption = rec["table_caption"]
    relation_raw = rec["relation_json"]

    relation = parse_relation(relation_raw)

    if relation is None:
        if len(failed_examples) < 5:
            failed_examples.append((webtable_id, str(relation_raw)[:300]))
        continue

    if not isinstance(relation, list) or len(relation) == 0:
        continue

    matrix = []
    for row in relation:
        if isinstance(row, list):
            matrix.append([safe_str(cell) for cell in row])

    if not matrix:
        continue

    # for vertical tables, transpose first
    if orientation == "VERTICAL":
        matrix = transpose_if_needed(matrix)

    header = None
    data_rows = matrix

    if bool(has_header) and len(matrix) >= 1:
        header = matrix[0]
        data_rows = matrix[1:]

    for row_idx, row in enumerate(data_rows):
        row_cells = [safe_str(x) for x in row]
        nonempty_cells = [x for x in row_cells if x != ""]

        if len(nonempty_cells) == 0:
            continue

        row_text = " | ".join(nonempty_cells)
        row_text_lc = row_text.lower()

        row_key = None
        try:
            if key_column_index is not None and int(key_column_index) >= 0 and int(key_column_index) < len(row_cells):
                row_key = safe_str(row_cells[int(key_column_index)])
        except Exception:
            row_key = None

        parsed_rows.append({
            "webtable_id": webtable_id,
            "source_domain": source_domain,
            "table_type": table_type,
            "orientation": orientation,
            "has_header": bool(has_header) if pd.notna(has_header) else None,
            "key_column_index": int(key_column_index) if pd.notna(key_column_index) else None,
            "page_title": page_title,
            "table_caption": table_caption,
            "header_json": json.dumps(header, ensure_ascii=False) if header is not None else None,
            "row_index": row_idx,
            "row_key": row_key,
            "row_json": json.dumps(row_cells, ensure_ascii=False),
            "row_text": row_text,
            "row_text_lc": row_text_lc,
            "cell_count": len(row_cells),
            "nonempty_cell_count": len(nonempty_cells)
        })

rows_df = pd.DataFrame(parsed_rows)

print(f"Parsed table rows: {len(rows_df):,}")

if failed_examples:
    print("\n--- Example failed relation_json values ---")
    for wt, ex in failed_examples:
        print(f"{wt}: {ex}\n")

if rows_df.empty:
    print("⚠️ Still no rows were parsed. Then we should rebuild wdc_webtables_raw_v1 differently.")
else:
    con.register("wdc_rows_df_view", rows_df)

    con.execute("""
        CREATE OR REPLACE TABLE wdc_music_table_rows_v1 AS
        SELECT *
        FROM wdc_rows_df_view
    """)

    print("✅ Table created: wdc_music_table_rows_v1")

    print("\n--- Row count ---")
    print(con.execute("""
        SELECT COUNT(*) AS n
        FROM wdc_music_table_rows_v1
    """).fetchdf())

    print("\n--- Distinct source tables ---")
    print(con.execute("""
        SELECT COUNT(DISTINCT webtable_id) AS n
        FROM wdc_music_table_rows_v1
    """).fetchdf())

    print("\n--- Orientation breakdown ---")
    print(con.execute("""
        SELECT orientation, COUNT(*) AS n
        FROM wdc_music_table_rows_v1
        GROUP BY orientation
        ORDER BY n DESC
    """).fetchdf())

    print("\n--- Preview ---")
    print(con.execute("""
        SELECT
            webtable_id,
            source_domain,
            table_type,
            orientation,
            row_index,
            row_key,
            row_text,
            cell_count,
            nonempty_cell_count
        FROM wdc_music_table_rows_v1
        LIMIT 30
    """).fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Loading strict candidate tables for robust relation parsing...
Loaded candidate tables: 935
Parsed table rows: 69

--- Example failed relation_json values ---
WT_30: [[hd, duration, published, description, name, type, category, Keywords, industryrating, agegate, agerequired, provider, language, subtitle, genres, actors, targetcountry, series, season, episode, characters, resolution, aspectratio, expirationdate, title, canembed, Source, pagecategories], [0, '00:1

WT_39: [[Created, Audio Files Included, Bit Rate, Sample Rate, Main Track Length, Tags], [13 February 14, WAV, 320 kbps, '16-Bit Stereo, 44.1 kHz', '0:04', 'access, alert, audio, brand, bright, chime, clean, company, corporate, effect, happy, ident, identify, intro, jingle, logo, modern, music, notificatio

WT_40: [[Created, Looped Audio, Audio Files Included, Bit Rate, Sample Rate, Main Track Length, 'Tempo (BPM)', Tags], [24 August 11, No, WAV, 320 kbps, '16-Bit Stereo, 44.1 kHz', '0:29', 115, 'action, action mov

In [22]:
import duckdb
import pandas as pd
import json
import re

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

sample_path = "../data_raw/webtables_sample/sample"

# ------------------------------------------------------------
# Step 12: Rebuild raw WDC table from the original sample file
# ------------------------------------------------------------
# Comments:
# - The previous raw table stored relation via CAST(... AS VARCHAR),
#   which broke JSON parsing for many rows.
# - Here we go back to the original sample file and read it line by line.
# - We preserve relation as valid JSON text using json.dumps().
# - This creates a clean raw staging table for the next WebTable steps.
print("Reading original WDC sample and creating wdc_webtables_raw_clean_v1...")

records = []

with open(sample_path, "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            obj = json.loads(line)
        except Exception:
            continue

        relation = obj.get("relation", None)

        records.append({
            "webtable_id": f"WT_{i}",
            "source_url": obj.get("url"),
            "page_title": obj.get("pageTitle"),
            "table_caption": obj.get("title"),
            "table_context_before": obj.get("textBeforeTable"),
            "table_context_after": obj.get("textAfterTable"),
            "orientation": obj.get("tableOrientation"),
            "has_header": obj.get("hasHeader"),
            "key_column_index": obj.get("keyColumnIndex"),
            "relation_json": json.dumps(relation, ensure_ascii=False) if relation is not None else None,
            "header_position": obj.get("headerPosition"),
            "table_type": obj.get("tableType"),
            "has_key_column": obj.get("hasKeyColumn"),
            "header_row_index": obj.get("headerRowIndex"),
            "table_num": obj.get("tableNum"),
            "s3_link": obj.get("s3Link"),
            "record_offset": obj.get("recordOffset"),
            "record_end_offset": obj.get("recordEndOffset"),
            "source_domain": None,
            "ingest_note": "wdc_sample_2015_clean"
        })

raw_df = pd.DataFrame(records)

# derive source_domain in pandas
def extract_domain(url):
    if pd.isna(url) or url is None:
        return None
    m = re.search(r"https?://([^/]+)", str(url))
    return m.group(1).lower() if m else None

raw_df["source_domain"] = raw_df["source_url"].apply(extract_domain)

print(f"Loaded raw tables: {len(raw_df):,}")

con.register("wdc_raw_clean_df_view", raw_df)

con.execute("""
    CREATE OR REPLACE TABLE wdc_webtables_raw_clean_v1 AS
    SELECT *
    FROM wdc_raw_clean_df_view
""")

print("✅ Table created: wdc_webtables_raw_clean_v1")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_webtables_raw_clean_v1
""").fetchdf())

print("\n--- Table type breakdown ---")
print(con.execute("""
    SELECT table_type, COUNT(*) AS n
    FROM wdc_webtables_raw_clean_v1
    GROUP BY table_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        table_type,
        orientation,
        has_header,
        has_key_column,
        page_title
    FROM wdc_webtables_raw_clean_v1
    LIMIT 20
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Reading original WDC sample and creating wdc_webtables_raw_clean_v1...


PermissionError: [Errno 13] Permission denied: '../data_raw/webtables_sample/sample'

In [23]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 13: Rebuild strict music-table candidates from clean raw data
# ------------------------------------------------------------
# Comments:
# - Same idea as before, but now based on the clean raw table.
# - This keeps only RELATION / ENTITY tables with stronger music signals.
# - We also exclude clearly irrelevant domains and page themes.
print("Creating wdc_music_tables_candidate_strict_v2...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_tables_candidate_strict_v2 AS
    WITH base AS (
        SELECT
            *,
            lower(coalesce(source_domain, '')) AS domain_lc,
            lower(coalesce(page_title, '')) AS page_title_lc,
            lower(coalesce(table_caption, '')) AS table_caption_lc,
            lower(coalesce(table_context_before, '')) AS ctx_before_lc,
            lower(coalesce(table_context_after, '')) AS ctx_after_lc,
            lower(coalesce(relation_json, '')) AS relation_lc
        FROM wdc_webtables_raw_clean_v1
    )
    SELECT *
    FROM base
    WHERE
        table_type IN ('RELATION', 'ENTITY')
        AND (
            regexp_matches(page_title_lc,
                'album|albums|discography|tracklist|track list|record label|music by|song|songs|single|singles|ep|lp|vinyl|cd|disc|artist|artists|band|bands|musician|musicians|soundtrack|release|releases|genre')
            OR regexp_matches(table_caption_lc,
                'album|albums|discography|tracklist|track list|record label|music by|song|songs|single|singles|ep|lp|vinyl|cd|disc|artist|artists|band|bands|soundtrack|release|genre')
            OR regexp_matches(relation_lc,
                'album|albums|track|tracks|tracklist|genre|label|artist|release|releases|discography|soundtrack')
        )
        AND NOT regexp_matches(domain_lc,
            'peoplefinders|healthgrades|stackoverflow|stackexchange|bccondos|clinicaltrials|cappex|dpreview|gilt|water\\.ca\\.gov|fantasysports|worldcat')
        AND NOT regexp_matches(page_title_lc,
            'box office|baseball|wiring diagram|materials and shaders|cross country|clinical trial|condo|wedding band|forum post|statistics')
""")

print("✅ Table created: wdc_music_tables_candidate_strict_v2")

print("\n--- Strict candidate count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_tables_candidate_strict_v2
""").fetchdf())

print("\n--- Table type breakdown ---")
print(con.execute("""
    SELECT table_type, COUNT(*) AS n
    FROM wdc_music_tables_candidate_strict_v2
    GROUP BY table_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Source domain preview ---")
print(con.execute("""
    SELECT source_domain, COUNT(*) AS n
    FROM wdc_music_tables_candidate_strict_v2
    GROUP BY source_domain
    ORDER BY n DESC
    LIMIT 20
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        table_type,
        orientation,
        page_title
    FROM wdc_music_tables_candidate_strict_v2
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_tables_candidate_strict_v2...


CatalogException: Catalog Error: Table with name wdc_webtables_raw_clean_v1 does not exist!
Did you mean "wdc_webtables_raw_v1"?

LINE 12:         FROM wdc_webtables_raw_clean_v1
                      ^

In [26]:
import duckdb
import pandas as pd
import json
import re
import pathlib
import builtins
import os

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 12 (fixed): Rebuild raw WDC table from the original sample file
# ------------------------------------------------------------
# Comments:
# - The previous attempt failed because the sample path could not be opened.
# - Here we:
#     1) verify whether the path is a file or directory
#     2) use builtins.open() instead of IPython's wrapped open()
#     3) preserve relation as valid JSON text with json.dumps()
print("Locating WDC sample file...")

sample_path = pathlib.Path("../data_raw/webtables_sample/sample")

# If "sample" is a directory, pick the first file inside it
if sample_path.exists() and sample_path.is_dir():
    files = [p for p in sample_path.iterdir() if p.is_file()]
    if not files:
        raise FileNotFoundError(f"No files found inside directory: {sample_path}")
    sample_path = files[0]

# If "sample" does not exist, show what's in the folder
if not sample_path.exists():
    folder = pathlib.Path("../data_raw/webtables_sample")
    print("❌ sample path not found.")
    print("Files currently in ../data_raw/webtables_sample:")
    if folder.exists():
        for p in folder.iterdir():
            print(" -", p.name, "(dir)" if p.is_dir() else "(file)")
    raise FileNotFoundError(f"Could not find sample file: {sample_path}")

print("Using sample file:", sample_path.resolve())

records = []

with builtins.open(sample_path, "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            obj = json.loads(line)
        except Exception:
            continue

        relation = obj.get("relation", None)

        records.append({
            "webtable_id": f"WT_{i}",
            "source_url": obj.get("url"),
            "page_title": obj.get("pageTitle"),
            "table_caption": obj.get("title"),
            "table_context_before": obj.get("textBeforeTable"),
            "table_context_after": obj.get("textAfterTable"),
            "orientation": obj.get("tableOrientation"),
            "has_header": obj.get("hasHeader"),
            "key_column_index": obj.get("keyColumnIndex"),
            "relation_json": json.dumps(relation, ensure_ascii=False) if relation is not None else None,
            "header_position": obj.get("headerPosition"),
            "table_type": obj.get("tableType"),
            "has_key_column": obj.get("hasKeyColumn"),
            "header_row_index": obj.get("headerRowIndex"),
            "table_num": obj.get("tableNum"),
            "s3_link": obj.get("s3Link"),
            "record_offset": obj.get("recordOffset"),
            "record_end_offset": obj.get("recordEndOffset"),
            "source_domain": None,
            "ingest_note": "wdc_sample_2015_clean"
        })

raw_df = pd.DataFrame(records)

def extract_domain(url):
    if pd.isna(url) or url is None:
        return None
    m = re.search(r"https?://([^/]+)", str(url))
    return m.group(1).lower() if m else None

raw_df["source_domain"] = raw_df["source_url"].apply(extract_domain)

print(f"Loaded raw tables: {len(raw_df):,}")

con.register("wdc_raw_clean_df_view", raw_df)

con.execute("""
    CREATE OR REPLACE TABLE wdc_webtables_raw_clean_v1 AS
    SELECT *
    FROM wdc_raw_clean_df_view
""")

print("✅ Table created: wdc_webtables_raw_clean_v1")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_webtables_raw_clean_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        table_type,
        orientation,
        has_header,
        has_key_column,
        page_title
    FROM wdc_webtables_raw_clean_v1
    LIMIT 10
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Locating WDC sample file...
Using sample file: C:\Users\User\OneDrive\Desktop\Eman_Thesis\data_raw\webtables_sample\sample\sample
Loaded raw tables: 9,275
✅ Table created: wdc_webtables_raw_clean_v1

--- Row count ---
      n
0  9275

--- Preview ---
  webtable_id            source_domain table_type orientation  has_header  \
0        WT_1                 1980s.fm     LAYOUT  HORIZONTAL       False   
1        WT_2                 1980s.fm     LAYOUT  HORIZONTAL       False   
2        WT_3              3docean.net     ENTITY    VERTICAL        True   
3        WT_4     67-72chevytrucks.com     LAYOUT    VERTICAL        True   
4        WT_5              965kvki.com   RELATION  HORIZONTAL        True   
5        WT_6  gayle.haarsma@dordt.edu   RELATION  HORIZONTAL        True   
6        WT_7         abacus.bates.edu   RELATION  HORIZONTAL        True   
7        WT_8            accleague.org     ENTITY    VERTICAL        True   
8        WT_9             aei.pitt.edu     E

In [27]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 13: Rebuild strict music-table candidates from clean raw data
# ------------------------------------------------------------
# Comments:
# - This now works only after wdc_webtables_raw_clean_v1 exists.
# - It filters the clean raw WDC sample down to stronger music-like tables.
print("Creating wdc_music_tables_candidate_strict_v2...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_tables_candidate_strict_v2 AS
    WITH base AS (
        SELECT
            *,
            lower(coalesce(source_domain, '')) AS domain_lc,
            lower(coalesce(page_title, '')) AS page_title_lc,
            lower(coalesce(table_caption, '')) AS table_caption_lc,
            lower(coalesce(table_context_before, '')) AS ctx_before_lc,
            lower(coalesce(table_context_after, '')) AS ctx_after_lc,
            lower(coalesce(relation_json, '')) AS relation_lc
        FROM wdc_webtables_raw_clean_v1
    )
    SELECT *
    FROM base
    WHERE
        table_type IN ('RELATION', 'ENTITY')
        AND (
            regexp_matches(page_title_lc,
                'album|albums|discography|tracklist|track list|record label|music by|song|songs|single|singles|ep|lp|vinyl|cd|disc|artist|artists|band|bands|musician|musicians|soundtrack|release|releases|genre')
            OR regexp_matches(table_caption_lc,
                'album|albums|discography|tracklist|track list|record label|music by|song|songs|single|singles|ep|lp|vinyl|cd|disc|artist|artists|band|bands|soundtrack|release|genre')
            OR regexp_matches(relation_lc,
                'album|albums|track|tracks|tracklist|genre|label|artist|release|releases|discography|soundtrack')
        )
        AND NOT regexp_matches(domain_lc,
            'peoplefinders|healthgrades|stackoverflow|stackexchange|bccondos|clinicaltrials|cappex|dpreview|gilt|water\\.ca\\.gov|fantasysports|worldcat')
        AND NOT regexp_matches(page_title_lc,
            'box office|baseball|wiring diagram|materials and shaders|cross country|clinical trial|condo|wedding band|forum post|statistics')
""")

print("✅ Table created: wdc_music_tables_candidate_strict_v2")

print("\n--- Strict candidate count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_tables_candidate_strict_v2
""").fetchdf())

print("\n--- Table type breakdown ---")
print(con.execute("""
    SELECT table_type, COUNT(*) AS n
    FROM wdc_music_tables_candidate_strict_v2
    GROUP BY table_type
    ORDER BY n DESC
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_tables_candidate_strict_v2...
✅ Table created: wdc_music_tables_candidate_strict_v2

--- Strict candidate count ---
     n
0  935

--- Table type breakdown ---
  table_type    n
0     ENTITY  547
1   RELATION  388

Connection closed ✅


In [28]:
import duckdb
import pandas as pd
import json

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 14: Parse row-level records from the clean relation field
# ------------------------------------------------------------
# Comments:
# - relation_json is now stored as valid JSON text in
#   wdc_webtables_raw_clean_v1 and carried into
#   wdc_music_tables_candidate_strict_v2.
# - We parse each candidate table into row-level records.
# - For VERTICAL tables, we transpose first so that downstream
#   matching sees row-like records.
# - If a header exists, we separate it from the data rows.
# - We keep:
#     * header_json
#     * row_json
#     * row_text
#     * row_key
#   so later steps can do schema matching and row-level ER.
print("Loading strict candidate tables from clean raw data...")

df = con.execute("""
    SELECT
        webtable_id,
        source_domain,
        table_type,
        orientation,
        has_header,
        key_column_index,
        page_title,
        table_caption,
        relation_json
    FROM wdc_music_tables_candidate_strict_v2
""").fetchdf()

print(f"Loaded candidate tables: {len(df):,}")

parsed_rows = []

def safe_str(x):
    return "" if x is None else str(x).strip()

def normalize_matrix(matrix):
    if not matrix:
        return []
    max_len = max(len(r) for r in matrix)
    return [r + [""] * (max_len - len(r)) for r in matrix]

def transpose_if_needed(matrix):
    if not matrix:
        return []
    padded = normalize_matrix(matrix)
    return [list(col) for col in zip(*padded)]

failed_tables = []

for _, rec in df.iterrows():
    webtable_id = rec["webtable_id"]
    source_domain = rec["source_domain"]
    table_type = rec["table_type"]
    orientation = safe_str(rec["orientation"]).upper()
    has_header = rec["has_header"]
    key_column_index = rec["key_column_index"]
    page_title = rec["page_title"]
    table_caption = rec["table_caption"]
    relation_json = rec["relation_json"]

    try:
        relation = json.loads(relation_json)
    except Exception:
        if len(failed_tables) < 10:
            failed_tables.append((webtable_id, "json_load_failed"))
        continue

    if not isinstance(relation, list) or len(relation) == 0:
        if len(failed_tables) < 10:
            failed_tables.append((webtable_id, "not_a_nonempty_list"))
        continue

    matrix = []
    for row in relation:
        if isinstance(row, list):
            matrix.append([safe_str(cell) for cell in row])

    if not matrix:
        if len(failed_tables) < 10:
            failed_tables.append((webtable_id, "no_list_rows"))
        continue

    # WDC tables can be column-major in some cases; we keep your earlier
    # thesis-safe rule:
    # - transpose VERTICAL tables first
    if orientation == "VERTICAL":
        matrix = transpose_if_needed(matrix)
    else:
        matrix = normalize_matrix(matrix)

    if not matrix:
        if len(failed_tables) < 10:
            failed_tables.append((webtable_id, "empty_after_normalization"))
        continue

    header = None
    data_rows = matrix

    if bool(has_header) and len(matrix) >= 1:
        header = [safe_str(x) for x in matrix[0]]
        data_rows = matrix[1:]

    # Skip tables with no usable data rows after header removal
    if len(data_rows) == 0:
        if len(failed_tables) < 10:
            failed_tables.append((webtable_id, "no_data_rows"))
        continue

    for row_idx, row in enumerate(data_rows):
        row_cells = [safe_str(x) for x in row]
        nonempty_cells = [x for x in row_cells if x != ""]

        if len(nonempty_cells) == 0:
            continue

        row_text = " | ".join(nonempty_cells)
        row_text_lc = row_text.lower()

        row_key = None
        try:
            if key_column_index is not None and pd.notna(key_column_index):
                k = int(key_column_index)
                if 0 <= k < len(row_cells):
                    row_key = safe_str(row_cells[k])
        except Exception:
            row_key = None

        parsed_rows.append({
            "webtable_id": webtable_id,
            "source_domain": source_domain,
            "table_type": table_type,
            "orientation": orientation,
            "has_header": bool(has_header) if pd.notna(has_header) else None,
            "key_column_index": int(key_column_index) if pd.notna(key_column_index) else None,
            "page_title": page_title,
            "table_caption": table_caption,
            "header_json": json.dumps(header, ensure_ascii=False) if header is not None else None,
            "row_index": row_idx,
            "row_key": row_key,
            "row_json": json.dumps(row_cells, ensure_ascii=False),
            "row_text": row_text,
            "row_text_lc": row_text_lc,
            "cell_count": len(row_cells),
            "nonempty_cell_count": len(nonempty_cells)
        })

rows_df = pd.DataFrame(parsed_rows)

print(f"Parsed table rows: {len(rows_df):,}")

if failed_tables:
    print("\n--- Example skipped / failed tables ---")
    for wt, reason in failed_tables:
        print(f"{wt}: {reason}")

if rows_df.empty:
    print("⚠️ No rows were parsed.")
else:
    con.register("wdc_rows_df_view_v2", rows_df)

    con.execute("""
        CREATE OR REPLACE TABLE wdc_music_table_rows_v2 AS
        SELECT *
        FROM wdc_rows_df_view_v2
    """)

    print("✅ Table created: wdc_music_table_rows_v2")

    print("\n--- Row count ---")
    print(con.execute("""
        SELECT COUNT(*) AS n
        FROM wdc_music_table_rows_v2
    """).fetchdf())

    print("\n--- Distinct source tables ---")
    print(con.execute("""
        SELECT COUNT(DISTINCT webtable_id) AS n
        FROM wdc_music_table_rows_v2
    """).fetchdf())

    print("\n--- Average rows per parsed table ---")
    print(con.execute("""
        SELECT ROUND(AVG(k), 2) AS avg_rows_per_table
        FROM (
            SELECT webtable_id, COUNT(*) AS k
            FROM wdc_music_table_rows_v2
            GROUP BY webtable_id
        )
    """).fetchdf())

    print("\n--- Orientation breakdown ---")
    print(con.execute("""
        SELECT orientation, COUNT(*) AS n
        FROM wdc_music_table_rows_v2
        GROUP BY orientation
        ORDER BY n DESC
    """).fetchdf())

    print("\n--- Preview ---")
    print(con.execute("""
        SELECT
            webtable_id,
            source_domain,
            table_type,
            orientation,
            row_index,
            row_key,
            row_text,
            cell_count,
            nonempty_cell_count
        FROM wdc_music_table_rows_v2
        LIMIT 30
    """).fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Loading strict candidate tables from clean raw data...
Loaded candidate tables: 935
Parsed table rows: 4,165
✅ Table created: wdc_music_table_rows_v2

--- Row count ---
      n
0  4165

--- Distinct source tables ---
     n
0  935

--- Average rows per parsed table ---
   avg_rows_per_table
0                4.45

--- Orientation breakdown ---
  orientation     n
0    VERTICAL  2606
1  HORIZONTAL  1559

--- Preview ---
   webtable_id                  source_domain table_type orientation  \
0        WT_30  arresteddevelopment.wikia.com     ENTITY    VERTICAL   
1        WT_30  arresteddevelopment.wikia.com     ENTITY    VERTICAL   
2        WT_30  arresteddevelopment.wikia.com     ENTITY    VERTICAL   
3        WT_30  arresteddevelopment.wikia.com     ENTITY    VERTICAL   
4        WT_30  arresteddevelopment.wikia.com     ENTITY    VERTICAL   
5        WT_30  arresteddevelopment.wikia.com     ENTITY    VERTICAL   
6        WT_30  arresteddevelopment.wikia.com     ENTITY    VE

In [29]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 15: Profile candidate WebTable schemas
# ------------------------------------------------------------
# Comments:
# - We inspect header_json plus row_text patterns to estimate whether
#   a table is structurally useful for music enrichment.
# - Goal:
#     identify tables that look like release/album metadata tables
#     rather than generic or technical pages.
# - We create binary schema signals such as:
#     * has_artist_signal
#     * has_title_signal
#     * has_year_signal
#     * has_genre_signal
#     * has_label_signal
#     * has_track_signal
print("Creating wdc_music_table_schema_profile_v1...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_table_schema_profile_v1 AS
    WITH row_agg AS (
        SELECT
            webtable_id,
            COUNT(*) AS row_count,
            MAX(CASE
                WHEN regexp_matches(lower(coalesce(row_text, '')),
                    'artist|artists|band|bands|musician|performed by|composer|singer')
                THEN 1 ELSE 0 END) AS row_artist_signal,

            MAX(CASE
                WHEN regexp_matches(lower(coalesce(row_text, '')),
                    'title|album|release|record|records|ep|lp|single|discography|song|songs')
                THEN 1 ELSE 0 END) AS row_title_signal,

            MAX(CASE
                WHEN regexp_matches(lower(coalesce(row_text, '')),
                    'year|released|release date|date')
                THEN 1 ELSE 0 END) AS row_year_signal,

            MAX(CASE
                WHEN regexp_matches(lower(coalesce(row_text, '')),
                    'genre|style')
                THEN 1 ELSE 0 END) AS row_genre_signal,

            MAX(CASE
                WHEN regexp_matches(lower(coalesce(row_text, '')),
                    'label|record label')
                THEN 1 ELSE 0 END) AS row_label_signal,

            MAX(CASE
                WHEN regexp_matches(lower(coalesce(row_text, '')),
                    'track|tracks|tracklist')
                THEN 1 ELSE 0 END) AS row_track_signal
        FROM wdc_music_table_rows_v2
        GROUP BY webtable_id
    ),
    base AS (
        SELECT
            t.webtable_id,
            t.source_domain,
            t.table_type,
            t.orientation,
            t.page_title,
            t.table_caption,
            lower(coalesce(r.header_json, '')) AS header_lc,
            lower(coalesce(t.page_title, '')) AS page_title_lc,
            lower(coalesce(t.table_caption, '')) AS table_caption_lc,
            a.row_count,
            a.row_artist_signal,
            a.row_title_signal,
            a.row_year_signal,
            a.row_genre_signal,
            a.row_label_signal,
            a.row_track_signal
        FROM wdc_music_tables_candidate_strict_v2 t
        LEFT JOIN (
            SELECT webtable_id, MIN(header_json) AS header_json
            FROM wdc_music_table_rows_v2
            GROUP BY webtable_id
        ) r
          ON t.webtable_id = r.webtable_id
        LEFT JOIN row_agg a
          ON t.webtable_id = a.webtable_id
    )
    SELECT
        *,

        CASE
            WHEN regexp_matches(header_lc,
                'artist|artists|band|bands|musician|performed by|composer|singer')
              OR regexp_matches(page_title_lc,
                'artist|band|musician|discography')
              OR coalesce(row_artist_signal, 0) = 1
            THEN 1 ELSE 0
        END AS has_artist_signal,

        CASE
            WHEN regexp_matches(header_lc,
                'title|album|release|record|records|ep|lp|single|discography|song|songs')
              OR regexp_matches(page_title_lc,
                'album|discography|release|record|song')
              OR coalesce(row_title_signal, 0) = 1
            THEN 1 ELSE 0
        END AS has_title_signal,

        CASE
            WHEN regexp_matches(header_lc,
                'year|released|release date|date')
              OR coalesce(row_year_signal, 0) = 1
            THEN 1 ELSE 0
        END AS has_year_signal,

        CASE
            WHEN regexp_matches(header_lc, 'genre|style')
              OR coalesce(row_genre_signal, 0) = 1
            THEN 1 ELSE 0
        END AS has_genre_signal,

        CASE
            WHEN regexp_matches(header_lc, 'label|record label')
              OR coalesce(row_label_signal, 0) = 1
            THEN 1 ELSE 0
        END AS has_label_signal,

        CASE
            WHEN regexp_matches(header_lc, 'track|tracks|tracklist')
              OR coalesce(row_track_signal, 0) = 1
            THEN 1 ELSE 0
        END AS has_track_signal
    FROM base
""")

print("✅ Table created: wdc_music_table_schema_profile_v1")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_schema_profile_v1
""").fetchdf())

print("\n--- Signal summary ---")
print(con.execute("""
    SELECT
        SUM(has_artist_signal) AS has_artist_signal,
        SUM(has_title_signal) AS has_title_signal,
        SUM(has_year_signal) AS has_year_signal,
        SUM(has_genre_signal) AS has_genre_signal,
        SUM(has_label_signal) AS has_label_signal,
        SUM(has_track_signal) AS has_track_signal
    FROM wdc_music_table_schema_profile_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        table_type,
        orientation,
        row_count,
        has_artist_signal,
        has_title_signal,
        has_year_signal,
        has_genre_signal,
        has_label_signal,
        has_track_signal,
        page_title
    FROM wdc_music_table_schema_profile_v1
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_schema_profile_v1...
✅ Table created: wdc_music_table_schema_profile_v1

--- Row count ---
     n
0  935

--- Signal summary ---
   has_artist_signal  has_title_signal  has_year_signal  has_genre_signal  \
0              102.0             430.0            288.0             160.0   

   has_label_signal  has_track_signal  
0              24.0             105.0  

--- Preview ---
   webtable_id                  source_domain table_type orientation  \
0        WT_30  arresteddevelopment.wikia.com     ENTITY    VERTICAL   
1        WT_39                audiojungle.net     ENTITY    VERTICAL   
2        WT_40                audiojungle.net     ENTITY    VERTICAL   
3        WT_41                audiojungle.net     ENTITY    VERTICAL   
4        WT_42                audiojungle.net     ENTITY    VERTICAL   
5        WT_43                audiojungle.net     ENTITY    VERTICAL   
6        WT_49         azmemory.azlibrary.gov     ENTITY    VERTICAL   
7     

In [30]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 16: Keep only schema-useful tables and their rows
# ------------------------------------------------------------
# Comments:
# - We now filter to tables that look structurally useful for
#   release-level or music-metadata enrichment.
# - Strong useful patterns:
#     * artist + title
#     * title + year
#     * title + genre
#     * title + label
#     * title + track info
# - This keeps the WebTable branch focused on plausible music metadata.
print("Creating wdc_music_table_schema_useful_v1 and wdc_music_table_rows_useful_v1...")

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_schema_useful_v1 AS
    SELECT *
    FROM wdc_music_table_schema_profile_v1
    WHERE
        (
            has_artist_signal = 1 AND has_title_signal = 1
        )
        OR (
            has_title_signal = 1 AND has_year_signal = 1
        )
        OR (
            has_title_signal = 1 AND has_genre_signal = 1
        )
        OR (
            has_title_signal = 1 AND has_label_signal = 1
        )
        OR (
            has_title_signal = 1 AND has_track_signal = 1
        )
""")

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_rows_useful_v1 AS
    SELECT r.*
    FROM wdc_music_table_rows_v2 r
    JOIN wdc_music_table_schema_useful_v1 s
      ON r.webtable_id = s.webtable_id
""")

print("✅ Tables created:")
print("   - wdc_music_table_schema_useful_v1")
print("   - wdc_music_table_rows_useful_v1")

print("\n--- Useful schema count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_schema_useful_v1
""").fetchdf())

print("\n--- Useful row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_rows_useful_v1
""").fetchdf())

print("\n--- Useful schema preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_count,
        has_artist_signal,
        has_title_signal,
        has_year_signal,
        has_genre_signal,
        has_label_signal,
        has_track_signal,
        page_title
    FROM wdc_music_table_schema_useful_v1
    LIMIT 30
""").fetchdf())

print("\n--- Useful row preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_index,
        row_key,
        row_text
    FROM wdc_music_table_rows_useful_v1
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_schema_useful_v1 and wdc_music_table_rows_useful_v1...
✅ Tables created:
   - wdc_music_table_schema_useful_v1
   - wdc_music_table_rows_useful_v1

--- Useful schema count ---
     n
0  304

--- Useful row count ---
      n
0  2100

--- Useful schema preview ---
   webtable_id                  source_domain  row_count  has_artist_signal  \
0        WT_30  arresteddevelopment.wikia.com         27                  0   
1        WT_40                audiojungle.net          7                  0   
2        WT_42                audiojungle.net          7                  0   
3        WT_49         azmemory.azlibrary.gov         15                  0   
4        WT_51         azmemory.azlibrary.gov          2                  0   
5        WT_52         azmemory.azlibrary.gov         20                  0   
6       WT_121    cdm15015.contentdm.oclc.org         18                  1   
7       WT_135    cdm15123.contentdm.oclc.org         16            

In [31]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 17: Keep only high-precision release-metadata WebTables
# ------------------------------------------------------------
# Comments:
# - The previous useful schema table is still too broad.
# - Here we keep only tables that look like real music release metadata
#   tables, not generic media / archive / technical pages.
# - We require stronger schema combinations and exclude noisy domains.
print("Creating wdc_music_table_schema_release_useful_v1...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_table_schema_release_useful_v1 AS
    WITH base AS (
        SELECT
            *,
            lower(coalesce(source_domain, '')) AS domain_lc,
            lower(coalesce(page_title, '')) AS page_title_lc
        FROM wdc_music_table_schema_profile_v1
    )
    SELECT *
    FROM base
    WHERE
        (
            has_artist_signal = 1 AND has_title_signal = 1
        )
        OR (
            has_artist_signal = 1 AND has_title_signal = 1 AND has_year_signal = 1
        )
        OR (
            has_artist_signal = 1 AND has_title_signal = 1 AND has_genre_signal = 1
        )
        OR (
            has_artist_signal = 1 AND has_title_signal = 1 AND has_label_signal = 1
        )
        OR (
            has_title_signal = 1 AND has_year_signal = 1 AND has_genre_signal = 1
        )
        OR (
            has_title_signal = 1 AND has_year_signal = 1 AND has_track_signal = 1
        )

        -- exclude obvious false-positive domains for this pilot
        AND NOT regexp_matches(domain_lc,
            'wikia|audiojungle|contentdm|cdmhost|azlibrary|cpa\\.state|city-data|investorplace|onthesnow|maps\\.|barmeister|contesting')

        -- exclude obvious non-release page themes
        AND NOT regexp_matches(page_title_lc,
            'season|episode|screenplay|historical|museum|student papers|artillery|school|weather|volatility|resort|magazine|directory|report')
""")

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_rows_release_useful_v1 AS
    SELECT r.*
    FROM wdc_music_table_rows_v2 r
    JOIN wdc_music_table_schema_release_useful_v1 s
      ON r.webtable_id = s.webtable_id
""")

print("✅ Tables created:")
print("   - wdc_music_table_schema_release_useful_v1")
print("   - wdc_music_table_rows_release_useful_v1")

print("\n--- Release-useful schema count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_schema_release_useful_v1
""").fetchdf())

print("\n--- Release-useful row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_rows_release_useful_v1
""").fetchdf())

print("\n--- Preview release-useful schemas ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_count,
        has_artist_signal,
        has_title_signal,
        has_year_signal,
        has_genre_signal,
        has_label_signal,
        has_track_signal,
        page_title
    FROM wdc_music_table_schema_release_useful_v1
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_schema_release_useful_v1...
✅ Tables created:
   - wdc_music_table_schema_release_useful_v1
   - wdc_music_table_rows_release_useful_v1

--- Release-useful schema count ---
     n
0  127

--- Release-useful row count ---
     n
0  934

--- Preview release-useful schemas ---
   webtable_id                          source_domain  row_count  \
0        WT_30          arresteddevelopment.wikia.com         27   
1        WT_52                 azmemory.azlibrary.gov         20   
2       WT_121            cdm15015.contentdm.oclc.org         18   
3       WT_135            cdm15123.contentdm.oclc.org         16   
4       WT_139            cdm15330.contentdm.oclc.org         15   
5       WT_147            cdm16007.contentdm.oclc.org          6   
6       WT_170            cdm16120.contentdm.oclc.org         22   
7       WT_171            cdm16280.contentdm.oclc.org         19   
8       WT_181                  cdm267401.cdmhost.com          8   
9       

In [32]:
import duckdb
import os
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

os.makedirs("../data_processed/exports", exist_ok=True)

# ------------------------------------------------------------
# Export high-precision WebTable pilot tables
# ------------------------------------------------------------
# Comments:
# - These exports capture the current best WebTable subset before
#   row-level entity matching.
# - We export:
#     1) schema-level useful tables
#     2) row-level useful rows
# - This is useful for:
#     * manual validation
#     * appendix material
#     * reproducibility in the thesis
print("Exporting release-useful WebTable pilot tables...")

schema_csv = "../data_processed/exports/wdc_music_table_schema_release_useful_v1.csv"
rows_csv = "../data_processed/exports/wdc_music_table_rows_release_useful_v1.csv"

con.execute(f"""
    COPY wdc_music_table_schema_release_useful_v1
    TO '{schema_csv}'
    WITH (HEADER, DELIMITER ',')
""")

con.execute(f"""
    COPY wdc_music_table_rows_release_useful_v1
    TO '{rows_csv}'
    WITH (HEADER, DELIMITER ',')
""")

print("✅ Exported:")
print(" -", schema_csv)
print(" -", rows_csv)

print("\n--- Export verification ---")
print(con.execute("""
    SELECT 'wdc_music_table_schema_release_useful_v1' AS table_name, COUNT(*) AS n
    FROM wdc_music_table_schema_release_useful_v1

    UNION ALL

    SELECT 'wdc_music_table_rows_release_useful_v1' AS table_name, COUNT(*) AS n
    FROM wdc_music_table_rows_release_useful_v1
""").fetchdf())

print("\n--- Preview: schema export ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_count,
        has_artist_signal,
        has_title_signal,
        has_year_signal,
        has_genre_signal,
        has_label_signal,
        has_track_signal,
        page_title
    FROM wdc_music_table_schema_release_useful_v1
    LIMIT 20
""").fetchdf())

print("\n--- Preview: row export ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_index,
        row_key,
        row_text
    FROM wdc_music_table_rows_release_useful_v1
    LIMIT 20
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Exporting release-useful WebTable pilot tables...
✅ Exported:
 - ../data_processed/exports/wdc_music_table_schema_release_useful_v1.csv
 - ../data_processed/exports/wdc_music_table_rows_release_useful_v1.csv

--- Export verification ---
                                 table_name    n
0  wdc_music_table_schema_release_useful_v1  127
1    wdc_music_table_rows_release_useful_v1  934

--- Preview: schema export ---
   webtable_id                          source_domain  row_count  \
0        WT_30          arresteddevelopment.wikia.com         27   
1        WT_52                 azmemory.azlibrary.gov         20   
2       WT_121            cdm15015.contentdm.oclc.org         18   
3       WT_135            cdm15123.contentdm.oclc.org         16   
4       WT_139            cdm15330.contentdm.oclc.org         15   
5       WT_147            cdm16007.contentdm.oclc.org          6   
6       WT_170            cdm16120.contentdm.oclc.org         22   
7       WT_171            cd

In [33]:
import duckdb
import os

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

os.makedirs("../data_processed/exports", exist_ok=True)

compact_csv = "../data_processed/exports/wdc_music_table_schema_release_useful_v1_compact.csv"

con.execute(f"""
    COPY (
        SELECT
            webtable_id,
            source_domain,
            row_count,
            has_artist_signal,
            has_title_signal,
            has_year_signal,
            has_genre_signal,
            has_label_signal,
            has_track_signal,
            page_title
        FROM wdc_music_table_schema_release_useful_v1
        ORDER BY source_domain, webtable_id
    )
    TO '{compact_csv}'
    WITH (HEADER, DELIMITER ',')
""")

print("✅ Exported compact schema file:", compact_csv)

con.close()
print("Connection closed ✅")

Connected ✅
✅ Exported compact schema file: ../data_processed/exports/wdc_music_table_schema_release_useful_v1_compact.csv
Connection closed ✅


In [34]:
import duckdb
import os
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

os.makedirs("../data_processed/exports", exist_ok=True)

# ------------------------------------------------------------
# Step 18: Create a manual review sample for schema auditing
# ------------------------------------------------------------
# Comments:
# - The current release-useful WebTable subset is still mixed:
#     * some true music release tables
#     * some music-related but not enrichment-useful tables
#     * some clear false positives
# - So before row-level matching, we create a manual audit sample.
# - This sample is for expert inspection and labeling in the thesis workflow.
print("Creating wdc_music_table_schema_review_sample_v1...")

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_schema_review_sample_v1 AS
    SELECT
        webtable_id,
        source_domain,
        row_count,
        has_artist_signal,
        has_title_signal,
        has_year_signal,
        has_genre_signal,
        has_label_signal,
        has_track_signal,
        page_title,
        table_caption,

        -- placeholders for manual review
        NULL::VARCHAR AS manual_label,
        NULL::VARCHAR AS review_note

    FROM wdc_music_table_schema_release_useful_v1
    ORDER BY
        source_domain,
        webtable_id
    LIMIT 100
""")

print("✅ Table created: wdc_music_table_schema_review_sample_v1")

print("\n--- Review sample size ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_schema_review_sample_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_count,
        has_artist_signal,
        has_title_signal,
        has_year_signal,
        has_genre_signal,
        has_label_signal,
        has_track_signal,
        page_title,
        manual_label,
        review_note
    FROM wdc_music_table_schema_review_sample_v1
    LIMIT 30
""").fetchdf())

# Export for manual annotation
out_path = "../data_processed/exports/wdc_music_table_schema_review_sample_v1.csv"

con.execute(f"""
    COPY wdc_music_table_schema_review_sample_v1
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"\n✅ Exported: {out_path}")

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_schema_review_sample_v1...
✅ Table created: wdc_music_table_schema_review_sample_v1

--- Review sample size ---
     n
0  100

--- Preview ---
   webtable_id                          source_domain  row_count  \
0        WT_30          arresteddevelopment.wikia.com         27   
1        WT_52                 azmemory.azlibrary.gov         20   
2       WT_121            cdm15015.contentdm.oclc.org         18   
3       WT_135            cdm15123.contentdm.oclc.org         16   
4       WT_139            cdm15330.contentdm.oclc.org         15   
5       WT_147            cdm16007.contentdm.oclc.org          6   
6       WT_170            cdm16120.contentdm.oclc.org         22   
7       WT_171            cdm16280.contentdm.oclc.org         19   
8       WT_181                  cdm267401.cdmhost.com          8   
9       WT_218                           cleorecs.com          5   
10      WT_233                collections.lib.uwm.edu         11   
11  

In [35]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Prefill manual labels for clearly identifiable review rows
# ------------------------------------------------------------
print("Updating wdc_music_table_schema_review_sample_v1...")

con.execute("""
    UPDATE wdc_music_table_schema_review_sample_v1
    SET
        manual_label = CASE webtable_id
            WHEN 'WT_30' THEN 'C'
            WHEN 'WT_52' THEN 'C'
            WHEN 'WT_121' THEN 'C'
            WHEN 'WT_135' THEN 'C'
            WHEN 'WT_139' THEN 'C'
            WHEN 'WT_147' THEN 'C'
            WHEN 'WT_170' THEN 'C'
            WHEN 'WT_171' THEN 'C'
            WHEN 'WT_181' THEN 'C'
            WHEN 'WT_218' THEN 'A'
            WHEN 'WT_233' THEN 'C'
            WHEN 'WT_268' THEN 'C'
            WHEN 'WT_272' THEN 'C'
            WHEN 'WT_277' THEN 'C'
            WHEN 'WT_278' THEN 'C'
            WHEN 'WT_279' THEN 'C'
            WHEN 'WT_494' THEN 'C'
            WHEN 'WT_505' THEN 'C'
            WHEN 'WT_518' THEN 'C'
            WHEN 'WT_531' THEN 'C'
            WHEN 'WT_534' THEN 'C'
            WHEN 'WT_535' THEN 'C'
            WHEN 'WT_539' THEN 'C'
            WHEN 'WT_587' THEN 'C'
            WHEN 'WT_593' THEN 'C'
            WHEN 'WT_617' THEN 'B'
            WHEN 'WT_618' THEN 'B'
            WHEN 'WT_619' THEN 'B'
            WHEN 'WT_877' THEN 'C'
            WHEN 'WT_917' THEN 'C'
            ELSE manual_label
        END,
        review_note = CASE webtable_id
            WHEN 'WT_30' THEN 'archive/media wiki page, not release metadata'
            WHEN 'WT_52' THEN 'magazine/archive page, not release metadata'
            WHEN 'WT_121' THEN 'archive collection page'
            WHEN 'WT_135' THEN 'city directory/archive page'
            WHEN 'WT_139' THEN 'school/public library archive page'
            WHEN 'WT_147' THEN 'newspaper/archive page'
            WHEN 'WT_170' THEN 'student newspaper archive'
            WHEN 'WT_171' THEN 'building/photo archive'
            WHEN 'WT_181' THEN 'historical newspaper archive'
            WHEN 'WT_218' THEN 'music label/store release page, useful'
            WHEN 'WT_233' THEN 'art image/collection page'
            WHEN 'WT_268' THEN 'protest/archive image page'
            WHEN 'WT_272' THEN 'art image collection'
            WHEN 'WT_277' THEN 'university newspaper archive'
            WHEN 'WT_278' THEN 'university newspaper archive'
            WHEN 'WT_279' THEN 'photograph collection page'
            WHEN 'WT_494' THEN 'architectural/photo archive'
            WHEN 'WT_505' THEN 'photo/history archive'
            WHEN 'WT_518' THEN 'digital library historical menu page'
            WHEN 'WT_531' THEN 'children literature archive'
            WHEN 'WT_534' THEN 'estate/photo archive'
            WHEN 'WT_535' THEN 'book jacket or art object page'
            WHEN 'WT_539' THEN 'government program/report page'
            WHEN 'WT_587' THEN 'archive admin page'
            WHEN 'WT_593' THEN 'historical mission/archive page'
            WHEN 'WT_617' THEN 'library catalog entry, music-related but weak'
            WHEN 'WT_618' THEN 'library catalog entry, music-related but weak'
            WHEN 'WT_619' THEN 'library catalog entry, music-related but weak'
            WHEN 'WT_877' THEN 'movie review or non-music page'
            WHEN 'WT_917' THEN 'forum false positive'
            ELSE review_note
        END
    WHERE webtable_id IN (
        'WT_30','WT_52','WT_121','WT_135','WT_139','WT_147','WT_170','WT_171',
        'WT_181','WT_218','WT_233','WT_268','WT_272','WT_277','WT_278','WT_279',
        'WT_494','WT_505','WT_518','WT_531','WT_534','WT_535','WT_539','WT_587',
        'WT_593','WT_617','WT_618','WT_619','WT_877','WT_917'
    )
""")

print("✅ Review labels updated")

print("\n--- Label summary so far ---")
print(con.execute("""
    SELECT manual_label, COUNT(*) AS n
    FROM wdc_music_table_schema_review_sample_v1
    GROUP BY manual_label
    ORDER BY manual_label
""").fetchdf())

print("\n--- Preview labeled rows ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        manual_label,
        review_note,
        page_title
    FROM wdc_music_table_schema_review_sample_v1
    WHERE manual_label IS NOT NULL
    ORDER BY webtable_id
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Updating wdc_music_table_schema_review_sample_v1...
✅ Review labels updated

--- Label summary so far ---
  manual_label   n
0            A   1
1            B   3
2            C  26
3         None  70

--- Preview labeled rows ---
   webtable_id                          source_domain manual_label  \
0       WT_121            cdm15015.contentdm.oclc.org            C   
1       WT_135            cdm15123.contentdm.oclc.org            C   
2       WT_139            cdm15330.contentdm.oclc.org            C   
3       WT_147            cdm16007.contentdm.oclc.org            C   
4       WT_170            cdm16120.contentdm.oclc.org            C   
5       WT_171            cdm16280.contentdm.oclc.org            C   
6       WT_181                  cdm267401.cdmhost.com            C   
7       WT_218                           cleorecs.com            A   
8       WT_233                collections.lib.uwm.edu            C   
9       WT_268                  contentdm.ad.umbc.edu    

In [36]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

out_path = "../data_processed/exports/wdc_music_table_schema_review_sample_v1_labeled.csv"

con.execute(f"""
    COPY wdc_music_table_schema_review_sample_v1
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print("✅ Exported:", out_path)

con.close()
print("Connection closed ✅")

Connected ✅
✅ Exported: ../data_processed/exports/wdc_music_table_schema_review_sample_v1_labeled.csv
Connection closed ✅


In [37]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 19: Build a stricter v2 release-useful WebTable subset
# ------------------------------------------------------------
# Comments:
# - v1 was still too noisy after schema profiling.
# - We now use the reviewed sample patterns to tighten the filter.
# - Strategy:
#     1) keep only strong schema combinations
#     2) exclude domain families that were repeatedly false positives
#     3) exclude archive / repository / image / newspaper / forum pages
#     4) keep a smaller, higher-precision set for row-level matching
print("Creating wdc_music_table_schema_release_useful_v2...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_table_schema_release_useful_v2 AS
    WITH base AS (
        SELECT
            *,
            lower(coalesce(source_domain, '')) AS domain_lc,
            lower(coalesce(page_title, '')) AS page_title_lc
        FROM wdc_music_table_schema_profile_v1
    )
    SELECT *
    FROM base
    WHERE
        -- ----------------------------------------------------
        -- Strong schema patterns only
        -- ----------------------------------------------------
        (
            has_artist_signal = 1 AND has_title_signal = 1
        )
        OR (
            has_artist_signal = 1 AND has_title_signal = 1 AND has_year_signal = 1
        )
        OR (
            has_artist_signal = 1 AND has_title_signal = 1 AND has_genre_signal = 1
        )
        OR (
            has_artist_signal = 1 AND has_title_signal = 1 AND has_label_signal = 1
        )

        -- ----------------------------------------------------
        -- Hard exclusions from reviewed false-positive families
        -- ----------------------------------------------------
        AND NOT regexp_matches(domain_lc,
            'wikia|gtaforums|audiojungle|contentdm|cdmhost|azlibrary|city-data|investorplace|onthesnow|maps\\.|contesting|digitalcollections|dig\\.library|earchives|econtent|familyeducation')

        -- ----------------------------------------------------
        -- Exclude archive / library / repository / document-like pages
        -- ----------------------------------------------------
        AND NOT regexp_matches(page_title_lc,
            'archive|historical|history|newspaper|student newspaper|photograph|directory|museum|school|magazine|report|board of trustees|digital library|art images|book jacket|mission|collection|collections|university photographs|civil war|frontier')

        -- ----------------------------------------------------
        -- Exclude obvious non-release media pages
        -- ----------------------------------------------------
        AND NOT regexp_matches(page_title_lc,
            'season|episode|screenplay|video trailer|movie review|weather|resort|forum|patent|board game|manga|visitors')
""")

print("✅ Table created: wdc_music_table_schema_release_useful_v2")

print("\nCreating wdc_music_table_rows_release_useful_v2...")

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_rows_release_useful_v2 AS
    SELECT r.*
    FROM wdc_music_table_rows_v2 r
    JOIN wdc_music_table_schema_release_useful_v2 s
      ON r.webtable_id = s.webtable_id
""")

print("✅ Table created: wdc_music_table_rows_release_useful_v2")

print("\n--- v2 schema count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_schema_release_useful_v2
""").fetchdf())

print("\n--- v2 row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_rows_release_useful_v2
""").fetchdf())

print("\n--- Source domain breakdown ---")
print(con.execute("""
    SELECT source_domain, COUNT(*) AS n
    FROM wdc_music_table_schema_release_useful_v2
    GROUP BY source_domain
    ORDER BY n DESC
    LIMIT 30
""").fetchdf())

print("\n--- Preview: v2 schemas ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_count,
        has_artist_signal,
        has_title_signal,
        has_year_signal,
        has_genre_signal,
        has_label_signal,
        has_track_signal,
        page_title
    FROM wdc_music_table_schema_release_useful_v2
    LIMIT 30
""").fetchdf())

print("\n--- Preview: v2 rows ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_index,
        row_key,
        row_text
    FROM wdc_music_table_rows_release_useful_v2
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_schema_release_useful_v2...
✅ Table created: wdc_music_table_schema_release_useful_v2

Creating wdc_music_table_rows_release_useful_v2...
✅ Table created: wdc_music_table_rows_release_useful_v2

--- v2 schema count ---
    n
0  75

--- v2 row count ---
     n
0  523

--- Source domain breakdown ---
                             source_domain  n
0                            gtaforums.com  8
1                         itunes.apple.com  5
2                           www.amoeba.com  4
3                         nl.wikipedia.org  3
4                  www.iraniantorrents.com  3
5                               zradio.org  2
6                              www.ebay.ca  2
7                  www.spirit-of-metal.com  2
8                    www.the-athenaeum.org  2
9                    www.boardgamegeek.com  2
10               merrick.library.miami.edu  2
11     publikationen.stub.uni-frankfurt.de  2
12                      www.metalstorm.net  2
13                 

In [41]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 20: Build a higher-precision v3 release-useful WebTable subset
# ------------------------------------------------------------
# Comments:
# - v2 still contains many false positives.
# - v3 adds domain-trust logic:
#     * keep trusted music-oriented domains
#     * exclude clearly noisy / irrelevant / archive-heavy domains
# - This is the expert high-precision subset for the first real
#   row-level WebTable matching experiment.
print("Creating wdc_music_table_schema_release_useful_v3...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_table_schema_release_useful_v3 AS
    WITH base AS (
        SELECT
            *,
            lower(coalesce(source_domain, '')) AS domain_lc,
            lower(coalesce(page_title, '')) AS page_title_lc
        FROM wdc_music_table_schema_profile_v1
    )
    SELECT *
    FROM base
    WHERE
        -- ----------------------------------------------------
        -- Strong release-oriented schema patterns
        -- ----------------------------------------------------
        (
        (
            has_artist_signal = 1 AND has_title_signal = 1
        )
        OR (
            has_artist_signal = 1 AND has_title_signal = 1 AND has_year_signal = 1
        )
        OR (
            has_artist_signal = 1 AND has_title_signal = 1 AND has_genre_signal = 1
        )
        OR (
            has_artist_signal = 1 AND has_title_signal = 1 AND has_label_signal = 1
        )
        )

        -- ----------------------------------------------------
        -- Keep only domains that are more plausible for music release data
        -- ----------------------------------------------------
        AND (
            regexp_matches(domain_lc,
                'itunes\\.apple\\.com|amoeba\\.com|recordheaven\\.net|metalstorm\\.net|spirit-of-metal\\.com|cleorecs\\.com|zradio\\.org|rrindex\\.com|wnur\\.org')
            OR regexp_matches(page_title_lc,
                'discography|album|albums|lp|ep|record|records|release|releases|tracklist|label|music')
        )

        -- ----------------------------------------------------
        -- Hard exclusions for known false-positive families
        -- ----------------------------------------------------
        AND NOT regexp_matches(domain_lc,
            'gtaforums|wikia|contentdm|cdmhost|digitalcollections|dig\\.library|econtent|earchives|library\\.|azlibrary|familyeducation|123helpme|teenink|boardgamegeek|vacationrentals|google\\.|patent|forums?|archive|history|museum|school|city-data|investorplace|onthesnow|maps\\.|contesting|orau|artsconnected|mosaic\\.cc\\.geneseo')

        -- ----------------------------------------------------
        -- Exclude clearly non-release page themes
        -- ----------------------------------------------------
        AND NOT regexp_matches(page_title_lc,
            'season|episode|screenplay|video trailer|movie review|weather|resort|forum|patent|board game|manga|visitors|historical|newspaper|student newspaper|photograph|directory|museum|school|magazine|report|art images|book jacket|civil war|frontier|mission|collection|collections')
""")

print("✅ Table created: wdc_music_table_schema_release_useful_v3")

print("\nCreating wdc_music_table_rows_release_useful_v3...")

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_rows_release_useful_v3 AS
    SELECT r.*
    FROM wdc_music_table_rows_v2 r
    JOIN wdc_music_table_schema_release_useful_v3 s
      ON r.webtable_id = s.webtable_id
""")

print("✅ Table created: wdc_music_table_rows_release_useful_v3")

print("\n--- v3 schema count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_schema_release_useful_v3
""").fetchdf())

print("\n--- v3 row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_rows_release_useful_v3
""").fetchdf())

print("\n--- Source domain breakdown ---")
print(con.execute("""
    SELECT source_domain, COUNT(*) AS n
    FROM wdc_music_table_schema_release_useful_v3
    GROUP BY source_domain
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: v3 schemas ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_count,
        has_artist_signal,
        has_title_signal,
        has_year_signal,
        has_genre_signal,
        has_label_signal,
        has_track_signal,
        page_title
    FROM wdc_music_table_schema_release_useful_v3
    LIMIT 30
""").fetchdf())

print("\n--- Preview: v3 rows ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_index,
        row_key,
        row_text
    FROM wdc_music_table_rows_release_useful_v3
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_schema_release_useful_v3...
✅ Table created: wdc_music_table_schema_release_useful_v3

Creating wdc_music_table_rows_release_useful_v3...
✅ Table created: wdc_music_table_rows_release_useful_v3

--- v3 schema count ---
    n
0  21

--- v3 row count ---
     n
0  103

--- Source domain breakdown ---
               source_domain  n
0           itunes.apple.com  5
1             www.amoeba.com  4
2    www.iraniantorrents.com  3
3                www.ebay.ca  2
4    www.spirit-of-metal.com  2
5               cleorecs.com  1
6      www.speed-n-power.com  1
7       digitalmedia.fws.gov  1
8  encore.skokielibrary.info  1
9            www.spotrac.com  1

--- Preview: v3 schemas ---
   webtable_id              source_domain  row_count  has_artist_signal  \
0       WT_218               cleorecs.com          5                  1   
1       WT_539       digitalmedia.fws.gov         18                  1   
2       WT_617  encore.skokielibrary.info          2     

In [42]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 21: Build a final high-precision v4 WebTable subset
# ------------------------------------------------------------
# Comments:
# - v3 is much better, but still contains a few clearly bad domains.
# - v4 is the first row-matching-ready subset.
# - Strategy:
#     * keep the v3 structure
#     * remove the remaining obvious false positives
#     * keep a small trusted pilot pool
print("Creating wdc_music_table_schema_release_useful_v4...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_table_schema_release_useful_v4 AS
    SELECT *
    FROM wdc_music_table_schema_release_useful_v3
    WHERE NOT regexp_matches(lower(coalesce(source_domain, '')),
        'gtaforums|spotrac|digitalmedia\.fws\.gov|encore\.skokielibrary|speed-n-power|iraniantorrents')
""")

print("✅ Table created: wdc_music_table_schema_release_useful_v4")

print("\nCreating wdc_music_table_rows_release_useful_v4...")

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_rows_release_useful_v4 AS
    SELECT r.*
    FROM wdc_music_table_rows_v2 r
    JOIN wdc_music_table_schema_release_useful_v4 s
      ON r.webtable_id = s.webtable_id
""")

print("✅ Table created: wdc_music_table_rows_release_useful_v4")

print("\n--- v4 schema count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_schema_release_useful_v4
""").fetchdf())

print("\n--- v4 row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_rows_release_useful_v4
""").fetchdf())

print("\n--- Source domain breakdown ---")
print(con.execute("""
    SELECT source_domain, COUNT(*) AS n
    FROM wdc_music_table_schema_release_useful_v4
    GROUP BY source_domain
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: v4 schemas ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_count,
        has_artist_signal,
        has_title_signal,
        has_year_signal,
        has_genre_signal,
        has_label_signal,
        has_track_signal,
        page_title
    FROM wdc_music_table_schema_release_useful_v4
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_schema_release_useful_v4...
✅ Table created: wdc_music_table_schema_release_useful_v4

Creating wdc_music_table_rows_release_useful_v4...
✅ Table created: wdc_music_table_rows_release_useful_v4

--- v4 schema count ---
    n
0  14

--- v4 row count ---
    n
0  59

--- Source domain breakdown ---
             source_domain  n
0         itunes.apple.com  5
1           www.amoeba.com  4
2              www.ebay.ca  2
3  www.spirit-of-metal.com  2
4             cleorecs.com  1

--- Preview: v4 schemas ---
   webtable_id            source_domain  row_count  has_artist_signal  \
0       WT_218             cleorecs.com          5                  1   
1      WT_2445           www.amoeba.com          2                  1   
2      WT_2446           www.amoeba.com          2                  1   
3      WT_2447           www.amoeba.com          4                  1   
4      WT_2448           www.amoeba.com          2                  1   
5      WT_3523    

In [43]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 21: Build a final high-precision v4 WebTable subset
# ------------------------------------------------------------
# Comments:
# - v3 is much better, but still contains a few clearly bad domains.
# - v4 is the first row-matching-ready subset.
# - Strategy:
#     * keep the v3 structure
#     * remove the remaining obvious false positives
#     * keep a small trusted pilot pool
print("Creating wdc_music_table_schema_release_useful_v4...")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_music_table_schema_release_useful_v4 AS
    SELECT *
    FROM wdc_music_table_schema_release_useful_v3
    WHERE NOT regexp_matches(lower(coalesce(source_domain, '')),
        'gtaforums|spotrac|digitalmedia\.fws\.gov|encore\.skokielibrary|speed-n-power|iraniantorrents')
""")

print("✅ Table created: wdc_music_table_schema_release_useful_v4")

print("\nCreating wdc_music_table_rows_release_useful_v4...")

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_rows_release_useful_v4 AS
    SELECT r.*
    FROM wdc_music_table_rows_v2 r
    JOIN wdc_music_table_schema_release_useful_v4 s
      ON r.webtable_id = s.webtable_id
""")

print("✅ Table created: wdc_music_table_rows_release_useful_v4")

print("\n--- v4 schema count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_schema_release_useful_v4
""").fetchdf())

print("\n--- v4 row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_rows_release_useful_v4
""").fetchdf())

print("\n--- Source domain breakdown ---")
print(con.execute("""
    SELECT source_domain, COUNT(*) AS n
    FROM wdc_music_table_schema_release_useful_v4
    GROUP BY source_domain
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: v4 schemas ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_count,
        has_artist_signal,
        has_title_signal,
        has_year_signal,
        has_genre_signal,
        has_label_signal,
        has_track_signal,
        page_title
    FROM wdc_music_table_schema_release_useful_v4
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_schema_release_useful_v4...
✅ Table created: wdc_music_table_schema_release_useful_v4

Creating wdc_music_table_rows_release_useful_v4...
✅ Table created: wdc_music_table_rows_release_useful_v4

--- v4 schema count ---
    n
0  14

--- v4 row count ---
    n
0  59

--- Source domain breakdown ---
             source_domain  n
0         itunes.apple.com  5
1           www.amoeba.com  4
2  www.spirit-of-metal.com  2
3              www.ebay.ca  2
4             cleorecs.com  1

--- Preview: v4 schemas ---
   webtable_id            source_domain  row_count  has_artist_signal  \
0       WT_218             cleorecs.com          5                  1   
1      WT_2445           www.amoeba.com          2                  1   
2      WT_2446           www.amoeba.com          2                  1   
3      WT_2447           www.amoeba.com          4                  1   
4      WT_2448           www.amoeba.com          2                  1   
5      WT_3523    

In [46]:
import duckdb
import pandas as pd
import re
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 22: First real row-level WebTable matching
# ------------------------------------------------------------
# Comments:
# - We match strong seeded music records against the v4 WebTable rows.
# - Core matching signals:
#     * artist similarity
#     * title similarity
#     * year compatibility
# - Extra support:
#     * barcode / UPC exact hit bonus when available
# - We do NOT use release_id as an external matching feature.
#   release_id is only for linking accepted evidence back to MB later.
print("Loading pilot records and v4 WebTable rows...")

pilot_df = con.execute("""
    SELECT
        base_record_key,
        base_record_status,
        release_id,
        external_record_id,
        seed_artist,
        seed_title,
        seed_year
    FROM music_enrichment_webtable_pilot_strong_v1
""").fetchdf()

rows_df = con.execute("""
    SELECT
        r.webtable_id,
        r.source_domain,
        r.page_title,
        r.row_index,
        r.row_key,
        r.row_text,
        r.row_text_lc,
        s.has_artist_signal,
        s.has_title_signal,
        s.has_year_signal,
        s.has_genre_signal,
        s.has_label_signal,
        s.has_track_signal
    FROM wdc_music_table_rows_release_useful_v4 r
    JOIN wdc_music_table_schema_release_useful_v4 s
      ON r.webtable_id = s.webtable_id
""").fetchdf()

# Pull barcode where available from the seeded base table
barcode_df = con.execute("""
    SELECT
        base_record_key,
        barcode
    FROM music_enrichment_base_for_webtables_seeded_v1
    WHERE barcode IS NOT NULL
""").fetchdf()

barcode_map = dict(zip(barcode_df["base_record_key"], barcode_df["barcode"]))

print(f"Loaded pilot records: {len(pilot_df):,}")
print(f"Loaded candidate rows: {len(rows_df):,}")

def norm_text(s):
    if s is None:
        return ""
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def norm_digits(s):
    if s is None:
        return ""
    return re.sub(r"\D+", "", str(s))

def find_barcode_like(text):
    if text is None:
        return None
    hits = re.findall(r"\b\d{12,14}\b", str(text))
    return hits[0] if hits else None

matches = []

for _, p in pilot_df.iterrows():
    base_record_key = p["base_record_key"]
    seed_artist = p["seed_artist"]
    seed_title = p["seed_title"]
    seed_year = p["seed_year"]
    seed_barcode = barcode_map.get(base_record_key)

    seed_artist_clean = norm_text(seed_artist)
    seed_title_clean = norm_text(seed_title)

    if not seed_artist_clean or not seed_title_clean:
        continue

    seed_artist_tokens = [t for t in seed_artist_clean.split() if len(t) >= 4]
    seed_title_tokens = [t for t in seed_title_clean.split() if len(t) >= 4]

    for _, r in rows_df.iterrows():
        row_text = r["row_text"]
        row_text_lc = r["row_text_lc"]

        if not row_text_lc:
            continue

        # ----------------------------------------------------
        # Coarse blocking:
        # require at least one strong token overlap from artist or title
        # ----------------------------------------------------
        coarse_hit = any(tok in row_text_lc for tok in seed_artist_tokens) or \
                     any(tok in row_text_lc for tok in seed_title_tokens)

        if not coarse_hit:
            continue

        row_clean = norm_text(row_text)

        # Similarities against row text
        artist_sim = fuzz.token_set_ratio(seed_artist_clean, row_clean)
        title_sim = fuzz.token_set_ratio(seed_title_clean, row_clean)

        # Year compatibility
        year_compat = 0.0
        if pd.notna(seed_year):
            year_str = str(int(seed_year))
            if year_str in row_text:
                year_compat = 100.0

        # Barcode / UPC bonus
        barcode_bonus = 0.0
        row_barcode = find_barcode_like(row_text)
        if seed_barcode is not None and row_barcode is not None:
            if norm_digits(seed_barcode) == norm_digits(row_barcode) and norm_digits(seed_barcode) != "":
                barcode_bonus = 20.0

        # Final row score
        row_match_score = (
            0.45 * artist_sim
            + 0.45 * title_sim
            + 0.10 * year_compat
            + barcode_bonus
        )

        matches.append({
            "base_record_key": base_record_key,
            "base_record_status": p["base_record_status"],
            "release_id": p["release_id"],
            "external_record_id": p["external_record_id"],
            "seed_artist": seed_artist,
            "seed_title": seed_title,
            "seed_year": seed_year,
            "seed_barcode": seed_barcode,
            "webtable_id": r["webtable_id"],
            "source_domain": r["source_domain"],
            "page_title": r["page_title"],
            "row_index": r["row_index"],
            "row_key": r["row_key"],
            "row_text": row_text,
            "artist_sim": artist_sim,
            "title_sim": title_sim,
            "year_compat": year_compat,
            "barcode_bonus": barcode_bonus,
            "row_match_score": row_match_score,
            "has_artist_signal": r["has_artist_signal"],
            "has_title_signal": r["has_title_signal"],
            "has_year_signal": r["has_year_signal"],
            "has_genre_signal": r["has_genre_signal"],
            "has_label_signal": r["has_label_signal"],
            "has_track_signal": r["has_track_signal"]
        })

match_df = pd.DataFrame(matches)

print(f"Raw row matches generated: {len(match_df):,}")

if match_df.empty:
    print("⚠️ No row matches found.")
else:
    # --------------------------------------------------------
    # Keep only strong first-pass matches
    # --------------------------------------------------------
    match_df = match_df[
        (
            (match_df["artist_sim"] >= 85) & (match_df["title_sim"] >= 85)
        )
        |
        (
            (match_df["artist_sim"] >= 90) & (match_df["title_sim"] >= 75)
        )
        |
        (
            (match_df["artist_sim"] >= 75) & (match_df["title_sim"] >= 90)
        )
        |
        (
            (match_df["barcode_bonus"] > 0) & (match_df["title_sim"] >= 70)
        )
    ].copy()

    print(f"Strong row matches retained: {len(match_df):,}")

    if match_df.empty:
        print("⚠️ No strong row matches survived the thresholds.")
    else:
        # Keep top 5 rows per seeded record
        match_df = (
            match_df.sort_values(
                ["base_record_key", "row_match_score", "title_sim", "artist_sim"],
                ascending=[True, False, False, False]
            )
            .groupby("base_record_key")
            .head(5)
            .copy()
        )

        con.register("wdc_row_matches_df_view", match_df)

        con.execute("""
            CREATE OR REPLACE TABLE wdc_music_table_row_matches_v1 AS
            SELECT *
            FROM wdc_row_matches_df_view
        """)

        print("✅ Table created: wdc_music_table_row_matches_v1")

        print("\n--- Row match count ---")
        print(con.execute("""
            SELECT COUNT(*) AS n
            FROM wdc_music_table_row_matches_v1
        """).fetchdf())

        print("\n--- Distinct seeded records with at least one match ---")
        print(con.execute("""
            SELECT COUNT(DISTINCT base_record_key) AS n
            FROM wdc_music_table_row_matches_v1
        """).fetchdf())

        print("\n--- Domain breakdown ---")
        print(con.execute("""
            SELECT source_domain, COUNT(*) AS n
            FROM wdc_music_table_row_matches_v1
            GROUP BY source_domain
            ORDER BY n DESC
        """).fetchdf())

        print("\n--- Preview ---")
        print(con.execute("""
            SELECT
                base_record_key,
                seed_artist,
                seed_title,
                seed_year,
                webtable_id,
                source_domain,
                row_index,
                row_text,
                artist_sim,
                title_sim,
                year_compat,
                barcode_bonus,
                row_match_score
            FROM wdc_music_table_row_matches_v1
            ORDER BY row_match_score DESC, base_record_key
            LIMIT 30
        """).fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Loading pilot records and v4 WebTable rows...
Loaded pilot records: 249
Loaded candidate rows: 59
Raw row matches generated: 267
Strong row matches retained: 0
⚠️ No strong row matches survived the thresholds.

Connection closed ✅


In [47]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 23: Build table-level record blocks from trusted WebTables
# ------------------------------------------------------------
# Comments:
# - Row-level matching was too strict because useful evidence is spread
#   across multiple rows of the same table.
# - So we aggregate all rows of each trusted WebTable into one combined
#   record block.
# - This block becomes the new matching unit for the first real
#   WebTable-to-seed entity resolution pass.
print("Creating wdc_music_table_record_blocks_v1...")

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_record_blocks_v1 AS
    WITH row_agg AS (
        SELECT
            webtable_id,
            source_domain,
            MIN(page_title) AS page_title,
            string_agg(coalesce(row_text, ''), ' || ' ORDER BY row_index) AS combined_row_text,
            string_agg(coalesce(row_key, ''), ' || ' ORDER BY row_index) AS combined_row_keys,
            COUNT(*) AS row_count
        FROM wdc_music_table_rows_release_useful_v4
        GROUP BY webtable_id, source_domain
    )
    SELECT
        a.webtable_id,
        a.source_domain,
        a.page_title,
        a.row_count,
        a.combined_row_keys,
        a.combined_row_text,
        lower(
            coalesce(a.page_title, '') || ' || ' ||
            coalesce(a.combined_row_keys, '') || ' || ' ||
            coalesce(a.combined_row_text, '')
        ) AS combined_block_text_lc
    FROM row_agg a
""")

print("✅ Table created: wdc_music_table_record_blocks_v1")

print("\n--- Block count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_record_blocks_v1
""").fetchdf())

print("\n--- Domain breakdown ---")
print(con.execute("""
    SELECT source_domain, COUNT(*) AS n
    FROM wdc_music_table_record_blocks_v1
    GROUP BY source_domain
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        row_count,
        page_title,
        combined_row_keys,
        combined_row_text
    FROM wdc_music_table_record_blocks_v1
    LIMIT 20
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_record_blocks_v1...
✅ Table created: wdc_music_table_record_blocks_v1

--- Block count ---
    n
0  14

--- Domain breakdown ---
             source_domain  n
0         itunes.apple.com  5
1           www.amoeba.com  4
2              www.ebay.ca  2
3  www.spirit-of-metal.com  2
4             cleorecs.com  1

--- Preview ---
   webtable_id            source_domain  row_count                                         page_title                                  combined_row_keys                                  combined_row_text
0      WT_2446           www.amoeba.com          2  Four Tet - As Serious As Your Life (Vinyl 12")...                                   Artist || Length  Artist | Four Tet , Jay Dee | Four Tet , Jay D...
1      WT_7139  www.spirit-of-metal.com          1  Condemnation (PL) - discography, line-up, biog...                              Others bands/comments  Others bands/comments | (R.I.P.) Betrayer (PL)...
2      WT_2445           

In [48]:
import duckdb
import pandas as pd
import re
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 24: Table-block entity matching for the WebTable branch
# ------------------------------------------------------------
# Comments:
# - We now match each seeded music record against an aggregated WebTable
#   block instead of a single row.
# - Core signals:
#     * artist similarity
#     * title similarity
#     * year compatibility
# - Support signal:
#     * barcode / UPC exact hit bonus
# - This is the correct matching unit for fragmented vertical/entity tables.
print("Loading seeded records and table blocks...")

pilot_df = con.execute("""
    SELECT
        base_record_key,
        base_record_status,
        release_id,
        external_record_id,
        seed_artist,
        seed_title,
        seed_year
    FROM music_enrichment_webtable_pilot_strong_v1
""").fetchdf()

block_df = con.execute("""
    SELECT
        webtable_id,
        source_domain,
        page_title,
        row_count,
        combined_row_keys,
        combined_row_text,
        combined_block_text_lc
    FROM wdc_music_table_record_blocks_v1
""").fetchdf()

barcode_df = con.execute("""
    SELECT
        base_record_key,
        barcode
    FROM music_enrichment_base_for_webtables_seeded_v1
    WHERE barcode IS NOT NULL
""").fetchdf()

barcode_map = dict(zip(barcode_df["base_record_key"], barcode_df["barcode"]))

print(f"Loaded seeded records: {len(pilot_df):,}")
print(f"Loaded table blocks: {len(block_df):,}")

def norm_text(s):
    if s is None:
        return ""
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def norm_digits(s):
    if s is None:
        return ""
    return re.sub(r"\D+", "", str(s))

def find_barcode_like(text):
    if text is None:
        return None
    hits = re.findall(r"\b\d{12,14}\b", str(text))
    return hits[0] if hits else None

matches = []

for _, p in pilot_df.iterrows():
    base_record_key = p["base_record_key"]
    seed_artist = p["seed_artist"]
    seed_title = p["seed_title"]
    seed_year = p["seed_year"]
    seed_barcode = barcode_map.get(base_record_key)

    seed_artist_clean = norm_text(seed_artist)
    seed_title_clean = norm_text(seed_title)

    if not seed_artist_clean or not seed_title_clean:
        continue

    seed_artist_tokens = [t for t in seed_artist_clean.split() if len(t) >= 4]
    seed_title_tokens = [t for t in seed_title_clean.split() if len(t) >= 4]

    for _, b in block_df.iterrows():
        block_text_lc = b["combined_block_text_lc"]
        if not block_text_lc:
            continue

        # ----------------------------------------------------
        # Coarse blocking:
        # require at least one strong artist token and one strong title token,
        # or a barcode hit
        # ----------------------------------------------------
        artist_token_hit = any(tok in block_text_lc for tok in seed_artist_tokens)
        title_token_hit = any(tok in block_text_lc for tok in seed_title_tokens)

        block_barcode = find_barcode_like(b["combined_row_text"])
        barcode_bonus = 0.0
        if seed_barcode is not None and block_barcode is not None:
            if norm_digits(seed_barcode) == norm_digits(block_barcode) and norm_digits(seed_barcode) != "":
                barcode_bonus = 20.0

        if not ((artist_token_hit and title_token_hit) or barcode_bonus > 0):
            continue

        block_clean = norm_text(
            str(b["page_title"]) + " || " + str(b["combined_row_text"])
        )

        artist_sim = fuzz.token_set_ratio(seed_artist_clean, block_clean)
        title_sim = fuzz.token_set_ratio(seed_title_clean, block_clean)

        year_compat = 0.0
        if pd.notna(seed_year):
            year_str = str(int(seed_year))
            if year_str in str(b["combined_row_text"]) or year_str in str(b["page_title"]):
                year_compat = 100.0

        block_match_score = (
            0.45 * artist_sim
            + 0.45 * title_sim
            + 0.10 * year_compat
            + barcode_bonus
        )

        matches.append({
            "base_record_key": base_record_key,
            "base_record_status": p["base_record_status"],
            "release_id": p["release_id"],
            "external_record_id": p["external_record_id"],
            "seed_artist": seed_artist,
            "seed_title": seed_title,
            "seed_year": seed_year,
            "seed_barcode": seed_barcode,
            "webtable_id": b["webtable_id"],
            "source_domain": b["source_domain"],
            "page_title": b["page_title"],
            "row_count": b["row_count"],
            "combined_row_keys": b["combined_row_keys"],
            "combined_row_text": b["combined_row_text"],
            "artist_sim": artist_sim,
            "title_sim": title_sim,
            "year_compat": year_compat,
            "barcode_bonus": barcode_bonus,
            "block_match_score": block_match_score
        })

match_df = pd.DataFrame(matches)

print(f"Raw block matches generated: {len(match_df):,}")

if match_df.empty:
    print("⚠️ No block matches found.")
else:
    # --------------------------------------------------------
    # Keep strong first-pass block matches
    # --------------------------------------------------------
    match_df = match_df[
        (
            (match_df["artist_sim"] >= 80) & (match_df["title_sim"] >= 80)
        )
        |
        (
            (match_df["artist_sim"] >= 88) & (match_df["title_sim"] >= 70)
        )
        |
        (
            (match_df["artist_sim"] >= 70) & (match_df["title_sim"] >= 88)
        )
        |
        (
            (match_df["barcode_bonus"] > 0) & (match_df["title_sim"] >= 65)
        )
    ].copy()

    print(f"Strong block matches retained: {len(match_df):,}")

    if match_df.empty:
        print("⚠️ No strong block matches survived the thresholds.")
    else:
        match_df = (
            match_df.sort_values(
                ["base_record_key", "block_match_score", "title_sim", "artist_sim"],
                ascending=[True, False, False, False]
            )
            .groupby("base_record_key")
            .head(5)
            .copy()
        )

        con.register("wdc_block_matches_df_view", match_df)

        con.execute("""
            CREATE OR REPLACE TABLE wdc_music_table_block_matches_v1 AS
            SELECT *
            FROM wdc_block_matches_df_view
        """)

        print("✅ Table created: wdc_music_table_block_matches_v1")

        print("\n--- Block match count ---")
        print(con.execute("""
            SELECT COUNT(*) AS n
            FROM wdc_music_table_block_matches_v1
        """).fetchdf())

        print("\n--- Distinct seeded records with at least one block match ---")
        print(con.execute("""
            SELECT COUNT(DISTINCT base_record_key) AS n
            FROM wdc_music_table_block_matches_v1
        """).fetchdf())

        print("\n--- Domain breakdown ---")
        print(con.execute("""
            SELECT source_domain, COUNT(*) AS n
            FROM wdc_music_table_block_matches_v1
            GROUP BY source_domain
            ORDER BY n DESC
        """).fetchdf())

        print("\n--- Preview ---")
        print(con.execute("""
            SELECT
                base_record_key,
                seed_artist,
                seed_title,
                seed_year,
                webtable_id,
                source_domain,
                page_title,
                artist_sim,
                title_sim,
                year_compat,
                barcode_bonus,
                block_match_score
            FROM wdc_music_table_block_matches_v1
            ORDER BY block_match_score DESC, base_record_key
            LIMIT 30
        """).fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Loading seeded records and table blocks...
Loaded seeded records: 249
Loaded table blocks: 14
Raw block matches generated: 22
Strong block matches retained: 0
⚠️ No strong block matches survived the thresholds.

Connection closed ✅


In [58]:
import duckdb
import pandas as pd
import re

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 25: Extract structured candidate fields from trusted tables
# ------------------------------------------------------------
# Comments:
# - Block matching showed that whole-table text is too noisy.
# - So we now extract field-like candidate text from page titles and row keys.
# - Goal:
#     build artist/title/year/barcode/label/genre candidates per webtable_id
print("Loading trusted v4 rows...")

rows_df = con.execute("""
    SELECT
        webtable_id,
        source_domain,
        page_title,
        row_index,
        row_key,
        row_text
    FROM wdc_music_table_rows_release_useful_v4
""").fetchdf()

def safe(s):
    return "" if s is None else str(s).strip()

def find_first_year(text):
    m = re.findall(r"\b(19\d{2}|20\d{2})\b", safe(text))
    return m[0] if m else None

def find_first_barcode(text):
    m = re.findall(r"\b\d{12,14}\b", safe(text))
    return m[0] if m else None

grouped = []

for webtable_id, g in rows_df.groupby("webtable_id"):
    source_domain = g["source_domain"].iloc[0]
    page_title = safe(g["page_title"].iloc[0])

    artist_bits = []
    title_bits = []
    year_bits = []
    barcode_bits = []
    label_bits = []
    genre_bits = []

    # --------------------------------------------------------
    # Page-title heuristics
    # --------------------------------------------------------
    page_title_lc = page_title.lower()

    # pattern: "TITLE by ARTIST"
    m_by = re.search(r"music\s*-\s*(.*?)\s+by\s+(.*)", page_title, flags=re.I)
    if m_by:
        title_bits.append(m_by.group(1).strip())
        artist_bits.append(m_by.group(2).strip())

    # pattern: "ARTIST - TITLE"
    m_dash = re.search(r"^(.*?)\s+-\s+(.*)$", page_title)
    if m_dash:
        left = m_dash.group(1).strip()
        right = m_dash.group(2).strip()
        if len(left) >= 2 and len(right) >= 2:
            artist_bits.append(left)
            title_bits.append(right)

    y = find_first_year(page_title)
    if y:
        year_bits.append(y)

    # --------------------------------------------------------
    # Row-level heuristics
    # --------------------------------------------------------
    for _, r in g.iterrows():
        row_key = safe(r["row_key"])
        row_text = safe(r["row_text"])
        row_key_lc = row_key.lower()
        row_text_lc = row_text.lower()

        if "artist" in row_key_lc:
            artist_bits.append(row_text)

        if "title" in row_key_lc or "name" in row_key_lc:
            title_bits.append(row_text)

        if "date" in row_key_lc or "year" in row_key_lc or "release" in row_key_lc:
            year_bits.append(row_text)

        if "upc" in row_key_lc or "ean" in row_key_lc or "barcode" in row_key_lc:
            barcode_bits.append(row_text)

        if "label" in row_key_lc:
            label_bits.append(row_text)

        if "genre" in row_key_lc or "style" in row_key_lc:
            genre_bits.append(row_text)

        # fallback barcode scan from row text
        bc = find_first_barcode(row_text)
        if bc:
            barcode_bits.append(bc)

    grouped.append({
        "webtable_id": webtable_id,
        "source_domain": source_domain,
        "page_title": page_title,
        "candidate_artist_text": " || ".join(dict.fromkeys([safe(x) for x in artist_bits if safe(x)])),
        "candidate_title_text": " || ".join(dict.fromkeys([safe(x) for x in title_bits if safe(x)])),
        "candidate_year_text": " || ".join(dict.fromkeys([safe(x) for x in year_bits if safe(x)])),
        "candidate_barcode_text": " || ".join(dict.fromkeys([safe(x) for x in barcode_bits if safe(x)])),
        "candidate_label_text": " || ".join(dict.fromkeys([safe(x) for x in label_bits if safe(x)])),
        "candidate_genre_text": " || ".join(dict.fromkeys([safe(x) for x in genre_bits if safe(x)])),
    })

field_df = pd.DataFrame(grouped)

con.register("wdc_candidate_fields_df_view", field_df)

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_candidate_fields_v1 AS
    SELECT *
    FROM wdc_candidate_fields_df_view
""")

print("✅ Table created: wdc_music_table_candidate_fields_v1")

print("\n--- Candidate field count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_candidate_fields_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        page_title,
        candidate_artist_text,
        candidate_title_text,
        candidate_year_text,
        candidate_barcode_text,
        candidate_label_text,
        candidate_genre_text
    FROM wdc_music_table_candidate_fields_v1
    LIMIT 20
""").fetchdf())


# Export for manual annotation
out_path = "../data_processed/exports/wdc_music_table_rows_release_useful_v4.csv"

con.execute(f"""
    COPY wdc_music_table_schema_review_sample_v1
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"\n✅ Exported: {out_path}")

con.close()
print("\nConnection closed ✅")

Connected ✅
Loading trusted v4 rows...
✅ Table created: wdc_music_table_candidate_fields_v1

--- Candidate field count ---
    n
0  14

--- Preview ---
   webtable_id            source_domain                                         page_title                              candidate_artist_text                               candidate_title_text candidate_year_text candidate_barcode_text candidate_label_text  \
0       WT_218             cleorecs.com  Dale Bozzio – New Wave Sessions (LP) | Cleopat...                                                                             Title | New Wave Sessions                                               Label | Cleopatra   
1      WT_2445           www.amoeba.com  Four Tet - As Serious As Your Life (Vinyl 12")...  Four Tet || Artist | Four Tet , Jay Dee , Guil...  As Serious As Your Life (Vinyl 12") - Amoeba M...                                                                   
2      WT_2446           www.amoeba.com  Four Tet - As Serious As Yo

In [52]:
import duckdb
import pandas as pd
import re
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 26: Field-aware matching of seeded records to WebTables
# ------------------------------------------------------------
# Comments:
# - We now compare seeded artist/title/year/barcode against extracted
#   candidate fields rather than full rows or whole blocks.
# - This is better suited to semi-structured music pages.
print("Loading seeded records and candidate fields...")

pilot_df = con.execute("""
    SELECT
        base_record_key,
        base_record_status,
        release_id,
        external_record_id,
        seed_artist,
        seed_title,
        seed_year
    FROM music_enrichment_webtable_pilot_strong_v1
""").fetchdf()

fields_df = con.execute("""
    SELECT *
    FROM wdc_music_table_candidate_fields_v1
""").fetchdf()

barcode_df = con.execute("""
    SELECT
        base_record_key,
        barcode
    FROM music_enrichment_base_for_webtables_seeded_v1
    WHERE barcode IS NOT NULL
""").fetchdf()

barcode_map = dict(zip(barcode_df["base_record_key"], barcode_df["barcode"]))

def norm_text(s):
    if s is None:
        return ""
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def norm_digits(s):
    if s is None:
        return ""
    return re.sub(r"\D+", "", str(s))

matches = []

for _, p in pilot_df.iterrows():
    seed_artist = p["seed_artist"]
    seed_title = p["seed_title"]
    seed_year = p["seed_year"]
    seed_barcode = barcode_map.get(p["base_record_key"])

    seed_artist_clean = norm_text(seed_artist)
    seed_title_clean = norm_text(seed_title)

    if not seed_artist_clean or not seed_title_clean:
        continue

    for _, f in fields_df.iterrows():
        cand_artist = f["candidate_artist_text"]
        cand_title = f["candidate_title_text"]
        cand_year = f["candidate_year_text"]
        cand_barcode = f["candidate_barcode_text"]

        artist_sim = fuzz.token_set_ratio(seed_artist_clean, norm_text(cand_artist))
        title_sim = fuzz.token_set_ratio(seed_title_clean, norm_text(cand_title))

        year_compat = 0.0
        if pd.notna(seed_year):
            if str(int(seed_year)) in safe(str(cand_year)):
                year_compat = 100.0

        barcode_bonus = 0.0
        if seed_barcode is not None and cand_barcode is not None:
            if norm_digits(seed_barcode) != "" and norm_digits(seed_barcode) in norm_digits(cand_barcode):
                barcode_bonus = 20.0

        field_match_score = (
            0.45 * artist_sim
            + 0.45 * title_sim
            + 0.10 * year_compat
            + barcode_bonus
        )

        matches.append({
            "base_record_key": p["base_record_key"],
            "base_record_status": p["base_record_status"],
            "release_id": p["release_id"],
            "external_record_id": p["external_record_id"],
            "seed_artist": seed_artist,
            "seed_title": seed_title,
            "seed_year": seed_year,
            "seed_barcode": seed_barcode,
            "webtable_id": f["webtable_id"],
            "source_domain": f["source_domain"],
            "page_title": f["page_title"],
            "candidate_artist_text": cand_artist,
            "candidate_title_text": cand_title,
            "candidate_year_text": cand_year,
            "candidate_barcode_text": cand_barcode,
            "candidate_label_text": f["candidate_label_text"],
            "candidate_genre_text": f["candidate_genre_text"],
            "artist_sim": artist_sim,
            "title_sim": title_sim,
            "year_compat": year_compat,
            "barcode_bonus": barcode_bonus,
            "field_match_score": field_match_score
        })

match_df = pd.DataFrame(matches)

print(f"Raw field matches generated: {len(match_df):,}")

if match_df.empty:
    print("⚠️ No field matches found.")
else:
    match_df = match_df[
        (
            (match_df["artist_sim"] >= 80) & (match_df["title_sim"] >= 80)
        )
        |
        (
            (match_df["artist_sim"] >= 90) & (match_df["title_sim"] >= 70)
        )
        |
        (
            (match_df["artist_sim"] >= 70) & (match_df["title_sim"] >= 90)
        )
        |
        (
            (match_df["barcode_bonus"] > 0) & (match_df["title_sim"] >= 60)
        )
    ].copy()

    print(f"Strong field matches retained: {len(match_df):,}")

    if match_df.empty:
        print("⚠️ No strong field matches survived the thresholds.")
    else:
        match_df = (
            match_df.sort_values(
                ["base_record_key", "field_match_score", "title_sim", "artist_sim"],
                ascending=[True, False, False, False]
            )
            .groupby("base_record_key")
            .head(5)
            .copy()
        )

        con.register("wdc_field_matches_df_view", match_df)

        con.execute("""
            CREATE OR REPLACE TABLE wdc_music_table_field_matches_v1 AS
            SELECT *
            FROM wdc_field_matches_df_view
        """)

        print("✅ Table created: wdc_music_table_field_matches_v1")

        print("\n--- Field match count ---")
        print(con.execute("""
            SELECT COUNT(*) AS n
            FROM wdc_music_table_field_matches_v1
        """).fetchdf())

        print("\n--- Distinct seeded records with at least one field match ---")
        print(con.execute("""
            SELECT COUNT(DISTINCT base_record_key) AS n
            FROM wdc_music_table_field_matches_v1
        """).fetchdf())

        print("\n--- Domain breakdown ---")
        print(con.execute("""
            SELECT source_domain, COUNT(*) AS n
            FROM wdc_music_table_field_matches_v1
            GROUP BY source_domain
            ORDER BY n DESC
        """).fetchdf())

        print("\n--- Preview ---")
        print(con.execute("""
            SELECT
                base_record_key,
                seed_artist,
                seed_title,
                seed_year,
                webtable_id,
                source_domain,
                artist_sim,
                title_sim,
                year_compat,
                barcode_bonus,
                field_match_score,
                candidate_artist_text,
                candidate_title_text,
                candidate_label_text,
                candidate_genre_text
            FROM wdc_music_table_field_matches_v1
            ORDER BY field_match_score DESC, base_record_key
            LIMIT 30
        """).fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Loading seeded records and candidate fields...
Raw field matches generated: 3,472
Strong field matches retained: 0
⚠️ No strong field matches survived the thresholds.

Connection closed ✅


In [56]:
import duckdb
import pandas as pd
import re

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 27: Clean extracted WebTable candidate fields
# ------------------------------------------------------------
# Comments:
# - v1 candidate fields still contain prefixes, store names, and noisy text.
# - We clean artist/title/year/barcode/label/genre into matching-ready fields.
print("Loading raw candidate fields...")

df = con.execute("""
    SELECT *
    FROM wdc_music_table_candidate_fields_v1
""").fetchdf()

def safe(s):
    return "" if s is None else str(s).strip()

def collapse_ws(s):
    return re.sub(r"\s+", " ", safe(s)).strip()

def strip_prefixes(s):
    s = safe(s)
    s = re.sub(r'^(artist|title|name|label|genre|style|record label|upc|ean|barcode)\s*[:|]\s*', '', s, flags=re.I)
    return collapse_ws(s)

def clean_artist(s):
    s = strip_prefixes(s)
    s = re.sub(r'\b(iTunes|Amoeba Music)\b', '', s, flags=re.I)
    s = re.split(r'\|\||\|', s)[0]
    s = re.sub(r'\s+', ' ', s).strip(" -|,")
    return collapse_ws(s)

def clean_title(s):
    s = strip_prefixes(s)
    s = re.sub(r'\bAmoeba Music\b', '', s, flags=re.I)
    s = re.sub(r'^\s*Music\s*-\s*', '', s, flags=re.I)
    s = re.sub(r'\s*-\s*Single\b', '', s, flags=re.I)
    s = re.sub(r'\s*-\s*Amoeba Music.*$', '', s, flags=re.I)
    s = re.sub(r'\(Vinyl.*?\)', '', s, flags=re.I)
    s = re.sub(r'\(MP3.*?\)', '', s, flags=re.I)
    s = re.sub(r'\(Remastered\)', '', s, flags=re.I)
    s = re.sub(r'\bdiscography,.*$', '', s, flags=re.I)
    s = re.split(r'\|\||\|', s)[0]
    s = re.sub(r'\s+', ' ', s).strip(" -|,")
    return collapse_ws(s)

def clean_year(s):
    m = re.findall(r'\b(19\d{2}|20\d{2})\b', safe(s))
    return m[0] if m else ""

def clean_barcode(s):
    m = re.findall(r'\b\d{12,14}\b', safe(s))
    return m[0] if m else ""

def clean_label(s):
    s = strip_prefixes(s)
    s = re.split(r'\|\||\|', s)[0]
    s = re.sub(r'^(label)\s*', '', s, flags=re.I)
    return collapse_ws(s).strip(" -|,")

def clean_genre(s):
    s = strip_prefixes(s)
    m = re.search(r'(dance|rock|metal|pop|jazz|classical|hip hop|hip-hop|electronic|folk|blues|country|soul|punk)', s, flags=re.I)
    return m.group(1) if m else ""

clean_rows = []

for _, row in df.iterrows():
    clean_rows.append({
        "webtable_id": row["webtable_id"],
        "source_domain": row["source_domain"],
        "page_title": row["page_title"],

        "candidate_artist_text": row["candidate_artist_text"],
        "candidate_title_text": row["candidate_title_text"],
        "candidate_year_text": row["candidate_year_text"],
        "candidate_barcode_text": row["candidate_barcode_text"],
        "candidate_label_text": row["candidate_label_text"],
        "candidate_genre_text": row["candidate_genre_text"],

        "candidate_artist_clean": clean_artist(row["candidate_artist_text"]),
        "candidate_title_clean": clean_title(row["candidate_title_text"]),
        "candidate_year_clean": clean_year(row["candidate_year_text"] or row["page_title"]),
        "candidate_barcode_clean": clean_barcode(row["candidate_barcode_text"]),
        "candidate_label_clean": clean_label(row["candidate_label_text"]),
        "candidate_genre_clean": clean_genre(row["candidate_genre_text"])
    })

clean_df = pd.DataFrame(clean_rows)

con.register("wdc_candidate_fields_clean_df_view", clean_df)

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_candidate_fields_clean_v1 AS
    SELECT *
    FROM wdc_candidate_fields_clean_df_view
""")

print("✅ Table created: wdc_music_table_candidate_fields_clean_v1")

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        candidate_artist_text,
        candidate_artist_clean,
        candidate_title_text,
        candidate_title_clean,
        candidate_year_clean,
        candidate_barcode_clean,
        candidate_label_clean,
        candidate_genre_clean
    FROM wdc_music_table_candidate_fields_clean_v1
    LIMIT 20
""").fetchdf())

# Export for manual annotation
out_path = "../data_processed/exports/wdc_music_table_candidate_fields_clean_v1.csv"

con.execute(f"""
    COPY wdc_music_table_schema_review_sample_v1
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"\n✅ Exported: {out_path}")

con.close()
print("\nConnection closed ✅")



Connected ✅
Loading raw candidate fields...
✅ Table created: wdc_music_table_candidate_fields_clean_v1

--- Preview ---
   webtable_id            source_domain                              candidate_artist_text                             candidate_artist_clean                               candidate_title_text                              candidate_title_clean  \
0       WT_218             cleorecs.com                                                                                                                                Title | New Wave Sessions                                  New Wave Sessions   
1      WT_2445           www.amoeba.com  Four Tet || Artist | Four Tet , Jay Dee , Guil...                                           Four Tet  As Serious As Your Life (Vinyl 12") - Amoeba M...                            As Serious As Your Life   
2      WT_2446           www.amoeba.com  Four Tet || Artist | Four Tet , Jay Dee | Four...                                           Four T

In [55]:
import duckdb
import pandas as pd
import re
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 28: Field-aware matching using cleaned candidate fields
# ------------------------------------------------------------
# Comments:
# - We now compare seeded records against cleaned WebTable fields.
# - This is the first real matching pass after cleaning artist/title/year/barcode.
# - Core signals:
#     * cleaned artist similarity
#     * cleaned title similarity
#     * year compatibility
# - Support signal:
#     * barcode exact hit bonus
print("Loading seeded records and cleaned candidate fields...")

pilot_df = con.execute("""
    SELECT
        base_record_key,
        base_record_status,
        release_id,
        external_record_id,
        seed_artist,
        seed_title,
        seed_year
    FROM music_enrichment_webtable_pilot_strong_v1
""").fetchdf()

fields_df = con.execute("""
    SELECT
        webtable_id,
        source_domain,
        page_title,
        candidate_artist_clean,
        candidate_title_clean,
        candidate_year_clean,
        candidate_barcode_clean,
        candidate_label_clean,
        candidate_genre_clean
    FROM wdc_music_table_candidate_fields_clean_v1
""").fetchdf()

barcode_df = con.execute("""
    SELECT
        base_record_key,
        barcode
    FROM music_enrichment_base_for_webtables_seeded_v1
    WHERE barcode IS NOT NULL
""").fetchdf()

barcode_map = dict(zip(barcode_df["base_record_key"], barcode_df["barcode"]))

def norm_text(s):
    if s is None:
        return ""
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def norm_digits(s):
    if s is None:
        return ""
    return re.sub(r"\D+", "", str(s))

matches = []

for _, p in pilot_df.iterrows():
    seed_artist = p["seed_artist"]
    seed_title = p["seed_title"]
    seed_year = p["seed_year"]
    seed_barcode = barcode_map.get(p["base_record_key"])

    seed_artist_clean = norm_text(seed_artist)
    seed_title_clean = norm_text(seed_title)

    if not seed_artist_clean or not seed_title_clean:
        continue

    for _, f in fields_df.iterrows():
        cand_artist = f["candidate_artist_clean"]
        cand_title = f["candidate_title_clean"]
        cand_year = f["candidate_year_clean"]
        cand_barcode = f["candidate_barcode_clean"]

        # skip candidates with no usable artist/title at all
        if not norm_text(cand_artist) and not norm_text(cand_title):
            continue

        artist_sim = fuzz.token_set_ratio(seed_artist_clean, norm_text(cand_artist))
        title_sim = fuzz.token_set_ratio(seed_title_clean, norm_text(cand_title))

        year_compat = 0.0
        if pd.notna(seed_year) and cand_year:
            if str(int(seed_year)) == str(cand_year):
                year_compat = 100.0
            elif abs(int(seed_year) - int(cand_year)) <= 1:
                year_compat = 80.0
            elif abs(int(seed_year) - int(cand_year)) <= 3:
                year_compat = 50.0

        barcode_bonus = 0.0
        if seed_barcode is not None and cand_barcode is not None:
            if norm_digits(seed_barcode) != "" and norm_digits(seed_barcode) == norm_digits(cand_barcode):
                barcode_bonus = 20.0

        field_match_score = (
            0.45 * artist_sim
            + 0.45 * title_sim
            + 0.10 * year_compat
            + barcode_bonus
        )

        matches.append({
            "base_record_key": p["base_record_key"],
            "base_record_status": p["base_record_status"],
            "release_id": p["release_id"],
            "external_record_id": p["external_record_id"],
            "seed_artist": seed_artist,
            "seed_title": seed_title,
            "seed_year": seed_year,
            "seed_barcode": seed_barcode,
            "webtable_id": f["webtable_id"],
            "source_domain": f["source_domain"],
            "page_title": f["page_title"],
            "candidate_artist_clean": cand_artist,
            "candidate_title_clean": cand_title,
            "candidate_year_clean": cand_year,
            "candidate_barcode_clean": cand_barcode,
            "candidate_label_clean": f["candidate_label_clean"],
            "candidate_genre_clean": f["candidate_genre_clean"],
            "artist_sim": artist_sim,
            "title_sim": title_sim,
            "year_compat": year_compat,
            "barcode_bonus": barcode_bonus,
            "field_match_score": field_match_score
        })

match_df = pd.DataFrame(matches)

print(f"Raw cleaned-field matches generated: {len(match_df):,}")

if match_df.empty:
    print("⚠️ No cleaned-field matches found.")
else:
    # --------------------------------------------------------
    # Keep strong but realistic first-pass matches
    # --------------------------------------------------------
    match_df = match_df[
        (
            (match_df["artist_sim"] >= 85) & (match_df["title_sim"] >= 85)
        )
        |
        (
            (match_df["artist_sim"] >= 92) & (match_df["title_sim"] >= 75)
        )
        |
        (
            (match_df["artist_sim"] >= 75) & (match_df["title_sim"] >= 92)
        )
        |
        (
            (match_df["barcode_bonus"] > 0) & (match_df["title_sim"] >= 55)
        )
    ].copy()

    print(f"Strong cleaned-field matches retained: {len(match_df):,}")

    if match_df.empty:
        print("⚠️ No strong cleaned-field matches survived the thresholds.")
    else:
        match_df = (
            match_df.sort_values(
                ["base_record_key", "field_match_score", "title_sim", "artist_sim"],
                ascending=[True, False, False, False]
            )
            .groupby("base_record_key")
            .head(5)
            .copy()
        )

        con.register("wdc_field_matches_clean_df_view", match_df)

        con.execute("""
            CREATE OR REPLACE TABLE wdc_music_table_field_matches_v2 AS
            SELECT *
            FROM wdc_field_matches_clean_df_view
        """)

        print("✅ Table created: wdc_music_table_field_matches_v2")

        print("\n--- Field match count ---")
        print(con.execute("""
            SELECT COUNT(*) AS n
            FROM wdc_music_table_field_matches_v2
        """).fetchdf())

        print("\n--- Distinct seeded records with at least one match ---")
        print(con.execute("""
            SELECT COUNT(DISTINCT base_record_key) AS n
            FROM wdc_music_table_field_matches_v2
        """).fetchdf())

        print("\n--- Domain breakdown ---")
        print(con.execute("""
            SELECT source_domain, COUNT(*) AS n
            FROM wdc_music_table_field_matches_v2
            GROUP BY source_domain
            ORDER BY n DESC
        """).fetchdf())

        print("\n--- Preview ---")
        print(con.execute("""
            SELECT
                base_record_key,
                seed_artist,
                seed_title,
                seed_year,
                webtable_id,
                source_domain,
                artist_sim,
                title_sim,
                year_compat,
                barcode_bonus,
                field_match_score,
                candidate_artist_clean,
                candidate_title_clean,
                candidate_label_clean,
                candidate_genre_clean
            FROM wdc_music_table_field_matches_v2
            ORDER BY field_match_score DESC, base_record_key
            LIMIT 30
        """).fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Loading seeded records and cleaned candidate fields...
Raw cleaned-field matches generated: 2,976
Strong cleaned-field matches retained: 0
⚠️ No strong cleaned-field matches survived the thresholds.

Connection closed ✅


In [59]:
import duckdb
import pandas as pd
import re
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 280)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Loading seeded records and cleaned candidate fields for diagnostics...")

pilot_df = con.execute("""
    SELECT
        base_record_key,
        base_record_status,
        release_id,
        external_record_id,
        seed_artist,
        seed_title,
        seed_year
    FROM music_enrichment_webtable_pilot_strong_v1
""").fetchdf()

fields_df = con.execute("""
    SELECT
        webtable_id,
        source_domain,
        page_title,
        candidate_artist_clean,
        candidate_title_clean,
        candidate_year_clean,
        candidate_barcode_clean,
        candidate_label_clean,
        candidate_genre_clean
    FROM wdc_music_table_candidate_fields_clean_v1
""").fetchdf()

barcode_df = con.execute("""
    SELECT
        base_record_key,
        barcode
    FROM music_enrichment_base_for_webtables_seeded_v1
    WHERE barcode IS NOT NULL
""").fetchdf()

barcode_map = dict(zip(barcode_df["base_record_key"], barcode_df["barcode"]))

def norm_text(s):
    if s is None:
        return ""
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def norm_digits(s):
    if s is None:
        return ""
    return re.sub(r"\D+", "", str(s))

rows = []

for _, p in pilot_df.iterrows():
    seed_artist_clean = norm_text(p["seed_artist"])
    seed_title_clean = norm_text(p["seed_title"])
    seed_year = p["seed_year"]
    seed_barcode = barcode_map.get(p["base_record_key"])

    if not seed_artist_clean or not seed_title_clean:
        continue

    for _, f in fields_df.iterrows():
        cand_artist = norm_text(f["candidate_artist_clean"])
        cand_title = norm_text(f["candidate_title_clean"])
        cand_year = f["candidate_year_clean"]
        cand_barcode = f["candidate_barcode_clean"]

        if not cand_artist and not cand_title:
            continue

        artist_sim = fuzz.token_set_ratio(seed_artist_clean, cand_artist)
        title_sim = fuzz.token_set_ratio(seed_title_clean, cand_title)

        year_compat = 0.0
        if pd.notna(seed_year) and cand_year:
            if str(int(seed_year)) == str(cand_year):
                year_compat = 100.0
            elif abs(int(seed_year) - int(cand_year)) <= 1:
                year_compat = 80.0
            elif abs(int(seed_year) - int(cand_year)) <= 3:
                year_compat = 50.0

        barcode_bonus = 0.0
        if seed_barcode is not None and cand_barcode:
            if norm_digits(seed_barcode) and norm_digits(seed_barcode) == norm_digits(cand_barcode):
                barcode_bonus = 20.0

        field_match_score = (
            0.45 * artist_sim
            + 0.45 * title_sim
            + 0.10 * year_compat
            + barcode_bonus
        )

        rows.append({
            "base_record_key": p["base_record_key"],
            "base_record_status": p["base_record_status"],
            "release_id": p["release_id"],
            "seed_artist": p["seed_artist"],
            "seed_title": p["seed_title"],
            "seed_year": p["seed_year"],
            "webtable_id": f["webtable_id"],
            "source_domain": f["source_domain"],
            "page_title": f["page_title"],
            "candidate_artist_clean": f["candidate_artist_clean"],
            "candidate_title_clean": f["candidate_title_clean"],
            "candidate_year_clean": f["candidate_year_clean"],
            "candidate_label_clean": f["candidate_label_clean"],
            "candidate_genre_clean": f["candidate_genre_clean"],
            "artist_sim": artist_sim,
            "title_sim": title_sim,
            "year_compat": year_compat,
            "barcode_bonus": barcode_bonus,
            "field_match_score": field_match_score
        })

diag_df = pd.DataFrame(rows)

print(f"Raw diagnostic pairs: {len(diag_df):,}")

if diag_df.empty:
    print("⚠️ No diagnostic pairs generated.")
else:
    diag_df = diag_df.sort_values(
        ["base_record_key", "field_match_score", "title_sim", "artist_sim"],
        ascending=[True, False, False, False]
    ).copy()

    # rank within each seed record
    diag_df["candidate_rank"] = (
        diag_df.groupby("base_record_key").cumcount() + 1
    )

    # keep top 5 per seed
    top_df = diag_df.groupby("base_record_key").head(5).copy()

    con.register("wdc_field_match_diagnostics_df_view", top_df)
    con.execute("""
        CREATE OR REPLACE TABLE wdc_music_table_field_match_diagnostics_v1 AS
        SELECT *
        FROM wdc_field_match_diagnostics_df_view
    """)

    print("✅ Table created: wdc_music_table_field_match_diagnostics_v1")

    print("\n--- Best-score summary across seeds ---")
    print(con.execute("""
        SELECT
            ROUND(AVG(field_match_score), 2) AS avg_score,
            ROUND(MAX(field_match_score), 2) AS max_score,
            ROUND(AVG(artist_sim), 2) AS avg_artist_sim,
            ROUND(MAX(artist_sim), 2) AS max_artist_sim,
            ROUND(AVG(title_sim), 2) AS avg_title_sim,
            ROUND(MAX(title_sim), 2) AS max_title_sim
        FROM (
            SELECT *
            FROM wdc_music_table_field_match_diagnostics_v1
            WHERE candidate_rank = 1
        )
    """).fetchdf())

    print("\n--- Domain breakdown for top-1 matches ---")
    print(con.execute("""
        SELECT source_domain, COUNT(*) AS n
        FROM wdc_music_table_field_match_diagnostics_v1
        WHERE candidate_rank = 1
        GROUP BY source_domain
        ORDER BY n DESC
    """).fetchdf())

    print("\n--- Top 30 matches overall ---")
    print(con.execute("""
        SELECT
            base_record_key,
            seed_artist,
            seed_title,
            seed_year,
            webtable_id,
            source_domain,
            candidate_artist_clean,
            candidate_title_clean,
            candidate_year_clean,
            candidate_label_clean,
            candidate_genre_clean,
            artist_sim,
            title_sim,
            year_compat,
            barcode_bonus,
            field_match_score,
            candidate_rank
        FROM wdc_music_table_field_match_diagnostics_v1
        ORDER BY field_match_score DESC, title_sim DESC, artist_sim DESC
        LIMIT 30
    """).fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Loading seeded records and cleaned candidate fields for diagnostics...
Raw diagnostic pairs: 2,976
✅ Table created: wdc_music_table_field_match_diagnostics_v1

--- Best-score summary across seeds ---
   avg_score  max_score  avg_artist_sim  max_artist_sim  avg_title_sim  max_title_sim
0       32.9       55.5           36.96           83.33          36.16          89.66

--- Domain breakdown for top-1 matches ---
      source_domain    n
0    www.amoeba.com  132
1  itunes.apple.com  116

--- Top 30 matches overall ---
   base_record_key                   seed_artist                                         seed_title  seed_year webtable_id     source_domain                             candidate_artist_clean                              candidate_title_clean candidate_year_clean  \
0       EXT_100072         various artistsry cd2    ministry of sound sessions eight todd terry cd2       1997     WT_8566  itunes.apple.com                                    Various Artists       

In [60]:
import duckdb
import pandas as pd
import os

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

export_dir = "../data_processed/exports/webtable_pipeline_exports_core"
os.makedirs(export_dir, exist_ok=True)

core_tables = [
    "music_enrichment_base_for_webtables_seeded_v1",
    "music_enrichment_webtable_pilot_strong_v1",
    "wdc_webtables_raw_clean_v1",
    "wdc_music_tables_candidate_strict_v2",
    "wdc_music_table_rows_v2",
    "wdc_music_table_schema_profile_v1",
    "wdc_music_table_schema_release_useful_v4",
    "wdc_music_table_rows_release_useful_v4",
    "wdc_music_table_record_blocks_v1",
    "wdc_music_table_candidate_fields_v1",
    "wdc_music_table_candidate_fields_clean_v1",
    "wdc_music_table_field_match_diagnostics_v1",
]

existing_df = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
""").fetchdf()

existing_tables = set(existing_df["table_name"].tolist())
core_tables = [t for t in core_tables if t in existing_tables]

manifest_rows = []

for table_name in core_tables:
    out_path = os.path.join(export_dir, f"{table_name}.csv").replace("\\", "/")
    row_count = con.execute(f"SELECT COUNT(*) AS n FROM {table_name}").fetchone()[0]

    con.execute(f"""
        COPY {table_name}
        TO '{out_path}'
        WITH (HEADER, DELIMITER ',')
    """)

    manifest_rows.append({
        "table_name": table_name,
        "row_count": row_count,
        "csv_path": out_path
    })

    print(f"✅ Exported {table_name} ({row_count:,} rows)")

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = os.path.join(export_dir, "webtable_pipeline_core_manifest.csv").replace("\\", "/")
manifest_df.to_csv(manifest_path, index=False)

print("\n--- Core export summary ---")
print(manifest_df)

con.close()
print("\nConnection closed ✅")

Connected ✅


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Exported music_enrichment_base_for_webtables_seeded_v1 (5,314,525 rows)
✅ Exported music_enrichment_webtable_pilot_strong_v1 (249 rows)
✅ Exported wdc_webtables_raw_clean_v1 (9,275 rows)
✅ Exported wdc_music_tables_candidate_strict_v2 (935 rows)
✅ Exported wdc_music_table_rows_v2 (4,165 rows)
✅ Exported wdc_music_table_schema_profile_v1 (935 rows)
✅ Exported wdc_music_table_schema_release_useful_v4 (14 rows)
✅ Exported wdc_music_table_rows_release_useful_v4 (59 rows)
✅ Exported wdc_music_table_record_blocks_v1 (14 rows)
✅ Exported wdc_music_table_candidate_fields_v1 (14 rows)
✅ Exported wdc_music_table_candidate_fields_clean_v1 (14 rows)
✅ Exported wdc_music_table_field_match_diagnostics_v1 (1,240 rows)

--- Core export summary ---
                                       table_name  row_count                                           csv_path
0   music_enrichment_base_for_webtables_seeded_v1    5314525  ../data_processed/exports/webtable_pipeline_ex...
1       music_enrichment_webtabl

In [61]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 29: Create a manual review set for top WebTable matches
# ------------------------------------------------------------
# Comments:
# - Automatic matching did not produce reliable accepted matches.
# - So we now create a focused manual evaluation sample from the
#   highest-scoring diagnostic candidates.
# - This lets us measure whether the best available WebTable matches are:
#     * correct
#     * partially relevant
#     * wrong
# - This is the right final evaluation step for the WebTable branch.
print("Creating wdc_music_table_field_match_review_top30_v1...")

con.execute("""
    CREATE OR REPLACE TABLE wdc_music_table_field_match_review_top30_v1 AS
    SELECT
        base_record_key,
        base_record_status,
        release_id,
        seed_artist,
        seed_title,
        seed_year,

        webtable_id,
        source_domain,
        page_title,

        candidate_artist_clean,
        candidate_title_clean,
        candidate_year_clean,
        candidate_label_clean,
        candidate_genre_clean,

        artist_sim,
        title_sim,
        year_compat,
        barcode_bonus,
        field_match_score,
        candidate_rank,

        NULL::VARCHAR AS manual_match_label,
        NULL::VARCHAR AS manual_review_note

    FROM wdc_music_table_field_match_diagnostics_v1
    ORDER BY
        field_match_score DESC,
        title_sim DESC,
        artist_sim DESC
    LIMIT 30
""")

print("✅ Table created: wdc_music_table_field_match_review_top30_v1")

print("\n--- Review sample size ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM wdc_music_table_field_match_review_top30_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        base_record_key,
        seed_artist,
        seed_title,
        webtable_id,
        source_domain,
        candidate_artist_clean,
        candidate_title_clean,
        candidate_year_clean,
        candidate_label_clean,
        candidate_genre_clean,
        artist_sim,
        title_sim,
        field_match_score,
        manual_match_label,
        manual_review_note
    FROM wdc_music_table_field_match_review_top30_v1
""").fetchdf())

# ------------------------------------------------------------
# Export for manual annotation
# ------------------------------------------------------------
out_path = "../data_processed/exports/wdc_music_table_field_match_review_top30_v1.csv"

con.execute(f"""
    COPY wdc_music_table_field_match_review_top30_v1
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"\n✅ Exported: {out_path}")

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating wdc_music_table_field_match_review_top30_v1...
✅ Table created: wdc_music_table_field_match_review_top30_v1

--- Review sample size ---
    n
0  30

--- Preview ---
   base_record_key                   seed_artist                                         seed_title webtable_id     source_domain                             candidate_artist_clean                              candidate_title_clean candidate_year_clean candidate_label_clean  \
0       EXT_100072         various artistsry cd2    ministry of sound sessions eight todd terry cd2     WT_8566  itunes.apple.com                                    Various Artists                   Red Hot + Rio 2 (Deluxe Edition)                                              
1       EXT_100072         various artistsry cd2    ministry of sound sessions eight todd terry cd2     WT_8559  itunes.apple.com                                    Various Artists     The greatest hits from 40's and 50's volume 24                           

In [62]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 30: Build a trusted known-release pilot for WebTable support
# ------------------------------------------------------------
# Comments:
# - We now stop asking WebTables to discover release identity from scratch.
# - Instead, we use already trusted enriched releases and test whether
#   WebTables can provide supporting metadata for them.
# - This is a more realistic and higher-value role for WebTables.
print("Creating music_webtable_support_pilot_v1...")

con.execute("""
    CREATE OR REPLACE TABLE music_webtable_support_pilot_v1 AS
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        release_id,
        release_title,
        main_artist_name,
        barcode,
        mb_year,
        enrichment_route,
        enrichment_status
    FROM thesis_musicbrainz_enrichment_master_v4
    WHERE enrichment_status = 'confirmed_mb_enrichment'
      AND release_id IS NOT NULL
      AND main_artist_name IS NOT NULL
      AND trim(main_artist_name) <> ''
      AND release_title IS NOT NULL
      AND trim(release_title) <> ''
""")

print("✅ Table created: music_webtable_support_pilot_v1")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM music_webtable_support_pilot_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        release_id,
        release_title,
        main_artist_name,
        barcode,
        mb_year
    FROM music_webtable_support_pilot_v1
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating music_webtable_support_pilot_v1...
✅ Table created: music_webtable_support_pilot_v1

--- Row count ---
      n
0  4214

--- Preview ---
    cds_id                artist                            title  cds_year  release_id                  release_title      main_artist_name        barcode  mb_year
0    10000       backstreet boys                       millennium      <NA>        8069                     Millennium       Backstreet Boys   012414167224     2016
1   100001               various      frankfurt trance vol 04 cd1      <NA>      190028               Frankfurt Trance       Various Artists  4017866122667     2019
2   100002             no return                  self mutilation      <NA>       12737                Self Mutilation             No Return  3760053840271     2020
3   100006      luv lite massive                         one love      <NA>     1032522                       One Love      Luv Lite Massive           None     2016
4   100008        

In [2]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 31: Create a manageable support pilot sample
# ------------------------------------------------------------
# Comments:
# - We start with a small trusted sample for manual and diagnostic work.
# - Prefer rows with barcode and year, because support evidence is easier
#   to evaluate when structured metadata exists.
print("Creating music_webtable_support_pilot_sample_v1...")

con.execute("""
    CREATE OR REPLACE TABLE music_webtable_support_pilot_sample_v1 AS
    SELECT *
    FROM music_webtable_support_pilot_v1
    ORDER BY
        CASE WHEN barcode IS NOT NULL AND trim(barcode) <> '' THEN 0 ELSE 1 END,
        CASE WHEN mb_year IS NOT NULL THEN 0 ELSE 1 END,
        release_id
    LIMIT 100
""")

print("✅ Table created: music_webtable_support_pilot_sample_v1")

print("\n--- Sample size ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM music_webtable_support_pilot_sample_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        cds_id,
        release_id,
        main_artist_name,
        release_title,
        barcode,
        mb_year
    FROM music_webtable_support_pilot_sample_v1
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")


Connected ✅
Creating music_webtable_support_pilot_sample_v1...
✅ Table created: music_webtable_support_pilot_sample_v1

--- Sample size ---
     n
0  100

--- Preview ---
    cds_id  release_id          main_artist_name  \
0   108407           7                 Tori Amos   
1   105777          14  Rage Against the Machine   
2     8717          51               Iron Maiden   
3     5664          51               Iron Maiden   
4   100198          89             Stacie Orrico   
5   106489          89             Stacie Orrico   
6   106702         138   Daryl Hall & John Oates   
7   106058         269               The Beatles   
8   104701         312               Evanescence   
9     6210         328                 Metallica   
10    3134         328                 Metallica   
11  104835         329                 Metallica   
12  106091         339                      Beck   
13  102391         471                     Sasha   
14  108196         488                 blink‐182 

In [3]:
import duckdb
import pandas as pd
import re
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 280)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Loading trusted support pilot and cleaned WebTable candidate fields...")

pilot_df = con.execute("""
    SELECT
        cds_id,
        release_id,
        main_artist_name,
        release_title,
        barcode,
        mb_year
    FROM music_webtable_support_pilot_sample_v1
""").fetchdf()

fields_df = con.execute("""
    SELECT
        webtable_id,
        source_domain,
        page_title,
        candidate_artist_clean,
        candidate_title_clean,
        candidate_year_clean,
        candidate_barcode_clean,
        candidate_label_clean,
        candidate_genre_clean
    FROM wdc_music_table_candidate_fields_clean_v1
""").fetchdf()

def norm_text(s):
    if s is None:
        return ""
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def norm_digits(s):
    if s is None:
        return ""
    return re.sub(r"\D+", "", str(s))

rows = []

for _, p in pilot_df.iterrows():
    artist_clean = norm_text(p["main_artist_name"])
    title_clean = norm_text(p["release_title"])
    year_val = p["mb_year"]
    barcode_val = p["barcode"]

    if not artist_clean or not title_clean:
        continue

    for _, f in fields_df.iterrows():
        cand_artist = norm_text(f["candidate_artist_clean"])
        cand_title = norm_text(f["candidate_title_clean"])
        cand_year = f["candidate_year_clean"]
        cand_barcode = f["candidate_barcode_clean"]

        if not cand_artist and not cand_title:
            continue

        artist_sim = fuzz.token_set_ratio(artist_clean, cand_artist)
        title_sim = fuzz.token_set_ratio(title_clean, cand_title)

        year_support = 0.0
        if pd.notna(year_val) and cand_year:
            if str(int(year_val)) == str(cand_year):
                year_support = 100.0
            elif abs(int(year_val) - int(cand_year)) <= 1:
                year_support = 80.0
            elif abs(int(year_val) - int(cand_year)) <= 3:
                year_support = 50.0

        barcode_support = 0.0
        if barcode_val is not None and cand_barcode:
            if norm_digits(barcode_val) and norm_digits(barcode_val) == norm_digits(cand_barcode):
                barcode_support = 100.0

        support_score = (
            0.40 * artist_sim
            + 0.40 * title_sim
            + 0.10 * year_support
            + 0.10 * barcode_support
        )

        rows.append({
            "cds_id": p["cds_id"],
            "release_id": p["release_id"],
            "main_artist_name": p["main_artist_name"],
            "release_title": p["release_title"],
            "barcode": p["barcode"],
            "mb_year": p["mb_year"],
            "webtable_id": f["webtable_id"],
            "source_domain": f["source_domain"],
            "page_title": f["page_title"],
            "candidate_artist_clean": f["candidate_artist_clean"],
            "candidate_title_clean": f["candidate_title_clean"],
            "candidate_year_clean": f["candidate_year_clean"],
            "candidate_barcode_clean": f["candidate_barcode_clean"],
            "candidate_label_clean": f["candidate_label_clean"],
            "candidate_genre_clean": f["candidate_genre_clean"],
            "artist_sim": artist_sim,
            "title_sim": title_sim,
            "year_support": year_support,
            "barcode_support": barcode_support,
            "support_score": support_score
        })

support_df = pd.DataFrame(rows)

print(f"Raw support pairs generated: {len(support_df):,}")

if support_df.empty:
    print("⚠️ No support pairs generated.")
else:
    support_df = (
        support_df.sort_values(
            ["release_id", "support_score", "title_sim", "artist_sim"],
            ascending=[True, False, False, False]
        )
        .groupby("release_id")
        .head(5)
        .copy()
    )

    con.register("music_webtable_support_df_view", support_df)

    con.execute("""
        CREATE OR REPLACE TABLE music_webtable_support_diagnostics_v1 AS
        SELECT *
        FROM music_webtable_support_df_view
    """)

    print("✅ Table created: music_webtable_support_diagnostics_v1")

    print("\n--- Support pair count ---")
    print(con.execute("""
        SELECT COUNT(*) AS n
        FROM music_webtable_support_diagnostics_v1
    """).fetchdf())

    print("\n--- Domain breakdown ---")
    print(con.execute("""
        SELECT source_domain, COUNT(*) AS n
        FROM music_webtable_support_diagnostics_v1
        GROUP BY source_domain
        ORDER BY n DESC
    """).fetchdf())

    print("\n--- Top support candidates ---")
    print(con.execute("""
        SELECT
            cds_id,
            release_id,
            main_artist_name,
            release_title,
            webtable_id,
            source_domain,
            candidate_artist_clean,
            candidate_title_clean,
            candidate_label_clean,
            candidate_genre_clean,
            artist_sim,
            title_sim,
            year_support,
            barcode_support,
            support_score
        FROM music_webtable_support_diagnostics_v1
        ORDER BY support_score DESC, title_sim DESC, artist_sim DESC
        LIMIT 30
    """).fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Loading trusted support pilot and cleaned WebTable candidate fields...
Raw support pairs generated: 1,200
✅ Table created: music_webtable_support_diagnostics_v1

--- Support pair count ---
     n
0  400

--- Domain breakdown ---
             source_domain    n
0           www.amoeba.com  228
1         itunes.apple.com  167
2  www.spirit-of-metal.com    3
3             cleorecs.com    2

--- Top support candidates ---
    cds_id  release_id        main_artist_name                          release_title webtable_id     source_domain candidate_artist_clean                           candidate_title_clean candidate_label_clean candidate_genre_clean  artist_sim   title_sim  year_support  \
0   104417         709             Twila Paris                          Greatest Hits     WT_8559  itunes.apple.com        Various Artists  The greatest hits from 40's and 50's volume 24                                               46.153846  100.000000           0.0   
1     8368        2783 

In [4]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 32: Build WebTable music release evidence table
# ------------------------------------------------------------
# Comments:
# - We do NOT start with matching anymore.
# - First, we treat WebTables as an external music evidence source.
# - Each retained trusted WebTable page contributes one extracted
#   music-evidence record.
# - This table is useful even when no MusicBrainz link is found.
print("Creating webtable_music_release_evidence_v1...")

con.execute("""
    CREATE OR REPLACE TABLE webtable_music_release_evidence_v1 AS
    SELECT
        webtable_id,
        source_domain,
        page_title,

        candidate_artist_clean AS web_artist,
        candidate_title_clean  AS web_title,
        candidate_year_clean   AS web_year,
        candidate_barcode_clean AS web_barcode,
        candidate_label_clean   AS web_label,
        candidate_genre_clean   AS web_genre,

        CASE WHEN candidate_artist_clean IS NOT NULL AND trim(candidate_artist_clean) <> '' THEN 1 ELSE 0 END AS has_artist,
        CASE WHEN candidate_title_clean  IS NOT NULL AND trim(candidate_title_clean)  <> '' THEN 1 ELSE 0 END AS has_title,
        CASE WHEN candidate_year_clean   IS NOT NULL AND trim(candidate_year_clean)   <> '' THEN 1 ELSE 0 END AS has_year,
        CASE WHEN candidate_barcode_clean IS NOT NULL AND trim(candidate_barcode_clean) <> '' THEN 1 ELSE 0 END AS has_barcode,
        CASE WHEN candidate_label_clean   IS NOT NULL AND trim(candidate_label_clean)   <> '' THEN 1 ELSE 0 END AS has_label,
        CASE WHEN candidate_genre_clean   IS NOT NULL AND trim(candidate_genre_clean)   <> '' THEN 1 ELSE 0 END AS has_genre

    FROM wdc_music_table_candidate_fields_clean_v1
""")

print("✅ Table created: webtable_music_release_evidence_v1")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM webtable_music_release_evidence_v1
""").fetchdf())

print("\n--- Evidence completeness summary ---")
print(con.execute("""
    SELECT
        SUM(has_artist) AS rows_with_artist,
        SUM(has_title) AS rows_with_title,
        SUM(has_year) AS rows_with_year,
        SUM(has_barcode) AS rows_with_barcode,
        SUM(has_label) AS rows_with_label,
        SUM(has_genre) AS rows_with_genre
    FROM webtable_music_release_evidence_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT *
    FROM webtable_music_release_evidence_v1
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating webtable_music_release_evidence_v1...
✅ Table created: webtable_music_release_evidence_v1

--- Row count ---
    n
0  14

--- Evidence completeness summary ---
   rows_with_artist  rows_with_title  rows_with_year  rows_with_barcode  rows_with_label  rows_with_genre
0              11.0             10.0             2.0                1.0              1.0              1.0

--- Preview ---
   webtable_id            source_domain                                         page_title                                         web_artist                                          web_title web_year   web_barcode  web_label web_genre  has_artist  has_title  has_year  \
0       WT_218             cleorecs.com  Dale Bozzio – New Wave Sessions (LP) | Cleopat...                                                                                     New Wave Sessions                         Cleopatra                     0          1         0   
1      WT_2445           www.amoeba.com  Fou

In [5]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 33: Score and classify WebTable music evidence quality
# ------------------------------------------------------------
# Comments:
# - We classify extracted WebTable evidence before any MusicBrainz matching.
# - This separates:
#     * strong release evidence
#     * partial release evidence
#     * weak music evidence
print("Creating webtable_music_release_evidence_scored_v1...")

con.execute("""
    CREATE OR REPLACE TABLE webtable_music_release_evidence_scored_v1 AS
    SELECT
        *,

        (
            2 * has_artist
          + 2 * has_title
          + 2 * has_barcode
          + 1 * has_year
          + 1 * has_label
          + 1 * has_genre
        ) AS evidence_score,

        CASE
            WHEN has_artist = 1 AND has_title = 1 AND has_barcode = 1
                THEN 'strong_release_evidence'
            WHEN has_artist = 1 AND has_title = 1 AND has_label = 1
                THEN 'strong_release_evidence'
            WHEN has_artist = 1 AND has_title = 1 AND has_year = 1
                THEN 'strong_release_evidence'
            WHEN has_artist = 1 AND has_title = 1
                THEN 'partial_release_evidence'
            WHEN has_artist = 1 OR has_title = 1
                THEN 'weak_music_evidence'
            ELSE 'nonusable'
        END AS evidence_class
    FROM webtable_music_release_evidence_v1
""")

print("✅ Table created: webtable_music_release_evidence_scored_v1")

print("\n--- Evidence class distribution ---")
print(con.execute("""
    SELECT evidence_class, COUNT(*) AS n
    FROM webtable_music_release_evidence_scored_v1
    GROUP BY evidence_class
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        web_artist,
        web_title,
        web_year,
        web_barcode,
        web_label,
        web_genre,
        evidence_score,
        evidence_class
    FROM webtable_music_release_evidence_scored_v1
    ORDER BY evidence_score DESC, webtable_id
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Creating webtable_music_release_evidence_scored_v1...
✅ Table created: webtable_music_release_evidence_scored_v1

--- Evidence class distribution ---
             evidence_class  n
0  partial_release_evidence  9
1       weak_music_evidence  3
2                 nonusable  2

--- Preview ---
   webtable_id            source_domain                                         web_artist                                          web_title web_year   web_barcode  web_label web_genre  evidence_score            evidence_class
0      WT_2445           www.amoeba.com                                           Four Tet                            As Serious As Your Life                                                           4  partial_release_evidence
1      WT_2446           www.amoeba.com                                           Four Tet                            As Serious As Your Life                                                           4  partial_release_evidence
2      WT_244

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 33: Score and classify WebTable music evidence quality
# ------------------------------------------------------------
# Comments:
# - We classify extracted WebTable evidence before any MusicBrainz matching.
# - This separates:
#     * strong release evidence
#     * partial release evidence
#     * weak music evidence
print("Creating webtable_music_release_evidence_scored_v1...")

con.execute("""
    CREATE OR REPLACE TABLE webtable_music_release_evidence_scored_v1 AS
    SELECT
        *,

        (
            2 * has_artist
          + 2 * has_title
          + 2 * has_barcode
          + 1 * has_year
          + 1 * has_label
          + 1 * has_genre
        ) AS evidence_score,

        CASE
            WHEN has_artist = 1 AND has_title = 1 AND has_barcode = 1
                THEN 'strong_release_evidence'
            WHEN has_artist = 1 AND has_title = 1 AND has_label = 1
                THEN 'strong_release_evidence'
            WHEN has_artist = 1 AND has_title = 1 AND has_year = 1
                THEN 'strong_release_evidence'
            WHEN has_artist = 1 AND has_title = 1
                THEN 'partial_release_evidence'
            WHEN has_artist = 1 OR has_title = 1
                THEN 'weak_music_evidence'
            ELSE 'nonusable'
        END AS evidence_class
    FROM webtable_music_release_evidence_v1
""")

print("✅ Table created: webtable_music_release_evidence_scored_v1")

print("\n--- Evidence class distribution ---")
print(con.execute("""
    SELECT evidence_class, COUNT(*) AS n
    FROM webtable_music_release_evidence_scored_v1
    GROUP BY evidence_class
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        webtable_id,
        source_domain,
        web_artist,
        web_title,
        web_year,
        web_barcode,
        web_label,
        web_genre,
        evidence_score,
        evidence_class
    FROM webtable_music_release_evidence_scored_v1
    ORDER BY evidence_score DESC, webtable_id
    LIMIT 30
""").fetchdf())

con.close()
print("\nConnection closed ✅")

In [ ]:
import duckdb
import pandas as pd
import re
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 34: Generate MusicBrainz candidate links from WebTable evidence
# ------------------------------------------------------------
# Comments:
# - Matching happens only AFTER extracting and classifying WebTable music evidence.
# - We use only strong and partial release evidence.
# - If no MusicBrainz match is found, the WebTable evidence remains valid
#   as unmatched external music evidence.
print("Loading WebTable evidence and MusicBrainz release data...")

web_df = con.execute("""
    SELECT
        webtable_id,
        source_domain,
        page_title,
        web_artist,
        web_title,
        web_year,
        web_barcode,
        web_label,
        web_genre,
        evidence_score,
        evidence_class
    FROM webtable_music_release_evidence_scored_v1
    WHERE evidence_class IN ('strong_release_evidence', 'partial_release_evidence')
""").fetchdf()

mb_df = con.execute("""
    SELECT
        release_id,
        release_title,
        main_artist_name,
        barcode,
        mb_year
    FROM mb_release_features
    WHERE main_artist_name IS NOT NULL
      AND trim(main_artist_name) <> ''
      AND release_title IS NOT NULL
      AND trim(release_title) <> ''
""").fetchdf()

print(f"Loaded WebTable evidence rows: {len(web_df):,}")
print(f"Loaded MB releases: {len(mb_df):,}")

def norm_text(s):
    if s is None:
        return ""
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def norm_digits(s):
    if s is None:
        return ""
    return re.sub(r"\D+", "", str(s))

rows = []

for _, w in web_df.iterrows():
    web_artist = norm_text(w["web_artist"])
    web_title = norm_text(w["web_title"])
    web_year = w["web_year"]
    web_barcode = w["web_barcode"]

    if not web_artist and not web_title:
        continue

    for _, m in mb_df.iterrows():
        mb_artist = norm_text(m["main_artist_name"])
        mb_title = norm_text(m["release_title"])
        mb_year = m["mb_year"]
        mb_barcode = m["barcode"]

        artist_sim = fuzz.token_set_ratio(web_artist, mb_artist) if web_artist and mb_artist else 0.0
        title_sim = fuzz.token_set_ratio(web_title, mb_title) if web_title and mb_title else 0.0

        year_support = 0.0
        if web_year is not None and str(web_year).strip() != "" and pd.notna(mb_year):
            try:
                wy = int(str(web_year))
                my = int(mb_year)
                if wy == my:
                    year_support = 100.0
                elif abs(wy - my) <= 1:
                    year_support = 80.0
                elif abs(wy - my) <= 3:
                    year_support = 50.0
            except:
                pass

        barcode_support = 0.0
        if norm_digits(web_barcode) and norm_digits(mb_barcode):
            if norm_digits(web_barcode) == norm_digits(mb_barcode):
                barcode_support = 100.0

        link_score = (
            0.40 * artist_sim
            + 0.40 * title_sim
            + 0.10 * year_support
            + 0.10 * barcode_support
        )

        rows.append({
            "webtable_id": w["webtable_id"],
            "source_domain": w["source_domain"],
            "page_title": w["page_title"],
            "web_artist": w["web_artist"],
            "web_title": w["web_title"],
            "web_year": w["web_year"],
            "web_barcode": w["web_barcode"],
            "web_label": w["web_label"],
            "web_genre": w["web_genre"],
            "evidence_score": w["evidence_score"],
            "evidence_class": w["evidence_class"],

            "release_id": m["release_id"],
            "main_artist_name": m["main_artist_name"],
            "release_title": m["release_title"],
            "mb_year": m["mb_year"],
            "mb_barcode": m["barcode"],

            "artist_sim": artist_sim,
            "title_sim": title_sim,
            "year_support": year_support,
            "barcode_support": barcode_support,
            "link_score": link_score
        })

link_df = pd.DataFrame(rows)

print(f"Raw WebTable→MB candidate links: {len(link_df):,}")

if link_df.empty:
    print("⚠️ No candidate links generated.")
else:
    link_df = (
        link_df.sort_values(
            ["webtable_id", "link_score", "title_sim", "artist_sim"],
            ascending=[True, False, False, False]
        )
        .groupby("webtable_id")
        .head(5)
        .copy()
    )

    con.register("webtable_mb_candidate_links_df_view", link_df)

    con.execute("""
        CREATE OR REPLACE TABLE webtable_to_musicbrainz_candidate_links_v1 AS
        SELECT *
        FROM webtable_mb_candidate_links_df_view
    """)

    print("✅ Table created: webtable_to_musicbrainz_candidate_links_v1")

    print("\n--- Candidate link count ---")
    print(con.execute("""
        SELECT COUNT(*) AS n
        FROM webtable_to_musicbrainz_candidate_links_v1
    """).fetchdf())

    print("\n--- Preview ---")
    print(con.execute("""
        SELECT
            webtable_id,
            source_domain,
            web_artist,
            web_title,
            release_id,
            main_artist_name,
            release_title,
            artist_sim,
            title_sim,
            year_support,
            barcode_support,
            link_score
        FROM webtable_to_musicbrainz_candidate_links_v1
        ORDER BY link_score DESC
        LIMIT 30
    """).fetchdf())

con.close()
print("\nConnection closed ✅")

Connected ✅
Loading WebTable evidence and MusicBrainz release data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded WebTable evidence rows: 9
Loaded MB releases: 5,309,878
